# DESeq2 Cross-Species Differential Accessibility (Pseudobulk)

**Goal:** Identify species-specific chromatin accessibility differences using donor-level pseudobulk replicates.

**Pipeline:**
1. Load pseudobulk counts from `12_fragment_matrices/` (output of notebook `08`)
2. Merge across species on shared peak IDs
3. Species-aware peak filtering (min samples per species, max count threshold)
4. Global VST + PCA for quality control
5. Per-cell-type DESeq2 with `~ 0 + species` design (all-vs-one contrasts)
6. Per-cell-type DESeq2 with `~ is_human` design (binary Human vs NonHuman)
7. Species-specific heatmaps and human-peak sharing analysis

**Input:** Parquet files from `cross_species_consensus_v3/12_fragment_matrices/{Species}/`
- `pseudobulk_counts.parquet` — peaks × pseudobulk samples
- `pseudobulk_groups.parquet` — sample metadata (cell_type, donor, label)

**Peak IDs:** Immutable names from BED column 4 (e.g. `unified_000001`, `human_peak_000003`).
After the peak_id fix in `src/fragment_matrices.py`, duplicates should no longer occur.

**Environment:** R kernel with DESeq2, arrow, ggplot2, pheatmap

In [1]:
# =============================================================================
# Cell 1: Load packages
# =============================================================================
suppressPackageStartupMessages({
  library(DESeq2)
  library(arrow)       # read parquet files
  library(ggplot2)
  library(pheatmap)
  library(dplyr)
  library(tibble)
  library(tidyr)
  library(readr)
})
message("All packages loaded")

# =============================================================================
# Cell 2: Configuration 
# =============================================================================
BASE <- "/cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks"

# ⬇️ UPDATED PATH: Pointing to the new fragment matrices folder
QUANT_DIR <- file.path(BASE, "cross_species_consensus_v3/12_fragment_matrices")

# We'll save the DESeq2 outputs to a new folder so we don't overwrite old stuff
OUT_DIR <- file.path(BASE, "cross_species_consensus_v3/13_deseq2_R_pseudobulk")
dir.create(OUT_DIR, showWarnings = FALSE, recursive = TRUE)

SPECIES <- c("Human", "Bonobo", "Chimpanzee", "Gorilla", "Macaque", "Marmoset")


All packages loaded



In [2]:
# =============================================================================
# Cell 3 & 4: Load and merge new Pseudobulk data & metadata
# =============================================================================
# NOTE: After the peak_id fix in src/fragment_matrices.py, region_ids in the
# parquet files are immutable peak names (e.g. "unified_000001") instead of
# coordinate strings. Coordinate-based duplicates (from liftover collisions)
# should no longer occur. The dedup block below is kept as a safety net.
# If you still see duplicates, re-run notebook 08 with FORCE_REBUILD = True.

all_counts <- list()
all_meta <- list()

for (sp in SPECIES) {
  sp_dir <- file.path(QUANT_DIR, sp)
  counts_file <- file.path(sp_dir, "pseudobulk_counts.parquet")
  meta_file <- file.path(sp_dir, "pseudobulk_groups.parquet") 
  
  if (!file.exists(counts_file)) {
    message("Skipping ", sp, ": files not found.")
    next
  }
  
  # Load counts
  counts <- as.data.frame(read_parquet(counts_file))
  
  # Safety net: deduplicate region_ids if any remain
  if (any(duplicated(counts$region_id))) {
    dup_ids <- unique(counts$region_id[duplicated(counts$region_id)])
    message("\n  ⚠️ Found ", length(dup_ids), " duplicated region IDs in ", sp, "!")
    
    # 1. INSPECT: Isolate all rows that share these duplicated IDs
    duplicate_df <- counts[counts$region_id %in% dup_ids, ]
    duplicate_df <- duplicate_df[order(duplicate_df$region_id), ]
    
    message("  🔍 Inspecting the first few columns of the duplicates:")
    print(duplicate_df[, 1:min(4, ncol(duplicate_df))])
    
    # Optional: Save them to a CSV so you can look at the full data
    inspect_file <- file.path(sp_dir, paste0("duplicated_regions_inspect.csv"))
    write.csv(duplicate_df, inspect_file, row.names = FALSE)
    message("  💾 Saved full duplicate dataframe to: ", inspect_file)
    
    # 2. FAST MERGE: Use base R's C-optimized rowsum instead of dplyr
    message("  ⚡ Fast-merging duplicates...")
    numeric_mat <- as.matrix(counts[, -which(names(counts) == "region_id")])
    merged_mat <- rowsum(numeric_mat, group = counts$region_id)
    
    # Rebuild the dataframe
    counts <- as.data.frame(merged_mat)
    counts$region_id <- rownames(counts)
  } else {
    rownames(counts) <- counts$region_id
  }
  
  counts$region_id <- NULL
  
  # Load metadata 
  meta <- as.data.frame(read_parquet(meta_file))
  
  # Prefix the sample labels with the species
  new_labels <- paste0(sp, "_", meta$label)
  colnames(counts) <- new_labels
  
  meta$sample_id <- new_labels
  meta$species <- sp
  
  all_counts[[sp]] <- counts
  all_meta[[sp]] <- meta
  message("Loaded ", sp, ": ", ncol(counts), " pseudobulk samples")
}

# 1. Inner join counts on shared peaks
shared_peaks <- Reduce(intersect, lapply(all_counts, rownames))
counts_merged <- do.call(cbind, unname(lapply(all_counts, function(x) x[shared_peaks, , drop = FALSE])))
counts_merged[is.na(counts_merged)] <- 0

# 2. Bind metadata 
meta_merged <- do.call(rbind, unname(all_meta))
rownames(meta_merged) <- meta_merged$sample_id

# 3. Ensure columns and rows align perfectly
counts_merged <- counts_merged[, rownames(meta_merged)]

message("\nMerged matrix: ", nrow(counts_merged), " peaks x ", ncol(counts_merged), " samples (donor pseudobulks)")

Loaded Human: 254 pseudobulk samples

Loaded Bonobo: 102 pseudobulk samples

Loaded Chimpanzee: 103 pseudobulk samples

Loaded Gorilla: 161 pseudobulk samples

Loaded Macaque: 82 pseudobulk samples

Loaded Marmoset: 112 pseudobulk samples


Merged matrix: 829088 peaks x 814 samples (donor pseudobulks)



In [3]:
head(meta_merged)

,cell_type,donor,region,age,label,n_cells,sample_id,species
,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>
Human_Adipocytes__B010__Colon__Adult,Adipocytes,B010,Colon,Adult,Adipocytes__B010__Colon__Adult,21,Human_Adipocytes__B010__Colon__Adult,Human
Human_Adipocytes__B012__Colon__Adult,Adipocytes,B012,Colon,Adult,Adipocytes__B012__Colon__Adult,16,Human_Adipocytes__B012__Colon__Adult,Human
Human_BEST4+_cells__B006__Colon__Adult,BEST4+ cells,B006,Colon,Adult,BEST4+_cells__B006__Colon__Adult,116,Human_BEST4+_cells__B006__Colon__Adult,Human
Human_BEST4+_cells__B006__Duodenum__Adult,BEST4+ cells,B006,Duodenum,Adult,BEST4+_cells__B006__Duodenum__Adult,24,Human_BEST4+_cells__B006__Duodenum__Adult,Human
Human_BEST4+_cells__B008__Colon__Adult,BEST4+ cells,B008,Colon,Adult,BEST4+_cells__B008__Colon__Adult,143,Human_BEST4+_cells__B008__Colon__Adult,Human
Human_BEST4+_cells__B008__Duodenum__Adult,BEST4+ cells,B008,Duodenum,Adult,BEST4+_cells__B008__Duodenum__Adult,28,Human_BEST4+_cells__B008__Duodenum__Adult,Human


In [4]:
# =============================================================================
# Helper Function: Aggregate Pseudobulk Samples
# =============================================================================
library(dplyr)

aggregate_pseudobulk <- function(counts_df, meta_df, merge_cols, sum_cols = c("n_cells")) {
  # 1. Identify which columns to group by 
  # Exclude columns we are merging, columns we are summing, and unique IDs
  exclude_cols <- c(merge_cols, sum_cols, "sample_id", "label", "total_counts", "agg_id")
  group_cols <- setdiff(colnames(meta_df), exclude_cols)
  
  message("Aggregating across: ", paste(merge_cols, collapse = ", "))
  message("Summing metadata: ", paste(sum_cols, collapse = ", "))
  message("Grouping samples by: ", paste(group_cols, collapse = ", "))
  
  # 2. Create a unique ID based on the grouping columns
  temp_meta <- meta_df[, group_cols, drop = FALSE]
  agg_id <- apply(temp_meta, 1, paste, collapse = "_")
  meta_df$agg_id <- agg_id
  
  # 3. Sum the matrix counts for identical grouping IDs
  message("  Summing matrix counts...")
  counts_t <- t(counts_df)
  merged_t <- rowsum(counts_t, group = agg_id)
  new_counts <- as.data.frame(t(merged_t))
  
  # 4. Rebuild metadata
  message("  Rebuilding metadata...")
  
  # A. Take the first instance of all non-summed columns
  new_meta_base <- meta_df %>%
    group_by(agg_id) %>%
    slice(1) %>%
    select(-any_of(sum_cols)) %>% 
    ungroup()
  
  # B. Sum the specified numeric columns (like n_cells)
  actual_sum_cols <- intersect(sum_cols, colnames(meta_df))
  if (length(actual_sum_cols) > 0) {
    new_meta_sums <- meta_df %>%
      group_by(agg_id) %>%
      summarize(across(all_of(actual_sum_cols), sum, na.rm = TRUE), .groups = "drop")
    
    # Merge base metadata with summed metadata
    new_meta <- left_join(new_meta_base, new_meta_sums, by = "agg_id")
  } else {
    new_meta <- new_meta_base
  }
  
  # 5. Overwrite the merged columns with "Merged"
  for (col in merge_cols) {
    if (col %in% colnames(new_meta)) {
      new_meta[[col]] <- as.character(new_meta[[col]])
      new_meta[[col]] <- "Merged"
    }
  }
  
  # 6. Clean up IDs
  new_meta$sample_id <- new_meta$agg_id
  new_meta$label <- new_meta$agg_id
  new_meta <- as.data.frame(new_meta)
  rownames(new_meta) <- new_meta$sample_id
  
  # 7. Ensure perfect alignment
  new_counts <- new_counts[, rownames(new_meta)]
  
  message("  Done! Reduced matrix from ", ncol(counts_df), " to ", ncol(new_counts), " samples.\n")
  return(list(counts = new_counts, meta = new_meta))
}

In [5]:
# =============================================================================
# Cell 4b: Modular Aggregation
# =============================================================================
COLS_TO_MERGE <- c("region")    # Columns to collapse
COLS_TO_SUM   <- c("n_cells")   # Metadata columns to add up
DO_AGGREGATE  <- TRUE           # Toggle aggregation on/off

# Filter for Adult samples ONLY (Do this BEFORE aggregation!)
meta_merged <- meta_merged[meta_merged$age == "Adult", ]
counts_merged <- counts_merged[, rownames(meta_merged)]
message("Subset to Adult samples: ", ncol(counts_merged), " samples remain.")

# Apply Modular Aggregation
if (DO_AGGREGATE && length(COLS_TO_MERGE) > 0) {
  agg_results <- aggregate_pseudobulk(
    counts_df = counts_merged, 
    meta_df = meta_merged, 
    merge_cols = COLS_TO_MERGE,
    sum_cols = COLS_TO_SUM
  )
  counts_merged <- agg_results$counts
  meta_merged <- agg_results$meta
}


Subset to Adult samples: 814 samples remain.

Aggregating across: region

Summing metadata: n_cells

Grouping samples by: cell_type, donor, age, species

  Summing matrix counts...

  Rebuilding metadata...

Warning message:
“There was 1 warning in `summarize()`.
ℹ In argument: `across(all_of(actual_sum_cols), sum, na.rm = TRUE)`.
ℹ In group 1: `agg_id = "Adipocytes_Almerida 17079_Adult_Marmoset"`.
Caused by warning:
! The `...` argument of `across()` is deprecated as of dplyr 1.1.0.
Supply arguments directly to `.fns` through an anonymous function instead.

  # Previously
  across(a:b, mean, na.rm = TRUE)

  # Now
  across(a:b, \(x) mean(x, na.rm = TRUE))”
  Done! Reduced matrix from 814 to 674 samples.




In [6]:
head(meta_merged)

,cell_type,donor,region,age,label,sample_id,species,agg_id,n_cells
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>
Adipocytes_Almerida 17079_Adult_Marmoset,Adipocytes,Almerida 17079,Merged,Adult,Adipocytes_Almerida 17079_Adult_Marmoset,Adipocytes_Almerida 17079_Adult_Marmoset,Marmoset,Adipocytes_Almerida 17079_Adult_Marmoset,42
Adipocytes_B010_Adult_Human,Adipocytes,B010,Merged,Adult,Adipocytes_B010_Adult_Human,Adipocytes_B010_Adult_Human,Human,Adipocytes_B010_Adult_Human,21
Adipocytes_B012_Adult_Human,Adipocytes,B012,Merged,Adult,Adipocytes_B012_Adult_Human,Adipocytes_B012_Adult_Human,Human,Adipocytes_B012_Adult_Human,16
Adipocytes_BN075_Adult_Chimpanzee,Adipocytes,BN075,Merged,Adult,Adipocytes_BN075_Adult_Chimpanzee,Adipocytes_BN075_Adult_Chimpanzee,Chimpanzee,Adipocytes_BN075_Adult_Chimpanzee,7
Adipocytes_BN085_Adult_Chimpanzee,Adipocytes,BN085,Merged,Adult,Adipocytes_BN085_Adult_Chimpanzee,Adipocytes_BN085_Adult_Chimpanzee,Chimpanzee,Adipocytes_BN085_Adult_Chimpanzee,17
Adipocytes_BN114_Adult_Bonobo,Adipocytes,BN114,Merged,Adult,Adipocytes_BN114_Adult_Bonobo,Adipocytes_BN114_Adult_Bonobo,Bonobo,Adipocytes_BN114_Adult_Bonobo,10


In [7]:
# =============================================================================
# Cell 5: Setup Experimental Design Metadata & Filter
# =============================================================================
library(dplyr)
library(ggplot2)

# Set Thresholds
MIN_SAMPLE_COUNTS <- 50000     
MIN_CELLS <- 100                 # Minimum cells required per pseudobulk

# 1. Calculate total counts (library size) per sample
meta_merged$total_counts <- colSums(counts_merged)

# 2. Print Top and Bottom 3 samples per Species
message("\n--- Top & Bottom 3 Samples by Total Counts per Species ---")
top_bottom_df <- meta_merged %>%
  group_by(species) %>%
  arrange(desc(total_counts)) %>%
  slice(c(1:3, (n()-2):n())) %>%
  mutate(Rank = c("Top 1", "Top 2", "Top 3", "Bottom 3", "Bottom 2", "Bottom 1")) %>%
  select(species, Rank, sample_id, total_counts, n_cells, cell_type) %>%
  ungroup()

print(as.data.frame(top_bottom_df))

# 3. Plot Total Counts Histogram with Threshold Line
plots_dir <- file.path(OUT_DIR, "plots")
dir.create(plots_dir, showWarnings = FALSE, recursive = TRUE)

hist_plot <- ggplot(meta_merged, aes(x = total_counts, fill = species)) +
  geom_histogram(bins = 30, color = "black", alpha = 0.8) +
  geom_vline(xintercept = MIN_SAMPLE_COUNTS, color = "red", linetype = "dashed", linewidth = 1) +
  facet_wrap(~ species, scales = "free_y") +
  theme_bw() +
  scale_x_log10(labels = scales::comma) + 
  labs(
    title = "Distribution of Total Counts per Pseudobulk Sample",
    subtitle = paste("Thresholds: Counts >=", MIN_SAMPLE_COUNTS, "| Cells >=", MIN_CELLS),
    x = "Total Counts (log10 scale)",
    y = "Number of Samples"
  ) +
  theme(legend.position = "none", axis.text.x = element_text(angle = 45, hjust = 1))

ggsave(file.path(plots_dir, "Count_Distributions_Per_Species_Log10.pdf"), hist_plot, width = 10, height = 6)
message("✅ Log-transformed count distribution plot saved.")

# 4. Apply dual filtering: Minimum counts AND Minimum cells
if ("n_cells" %in% colnames(meta_merged)) {
  keep_samples <- meta_merged$total_counts >= MIN_SAMPLE_COUNTS & meta_merged$n_cells >= MIN_CELLS
  message("\nFiltering: Requiring >= ", MIN_SAMPLE_COUNTS, " counts AND >= ", MIN_CELLS, " cells.")
} else {
  keep_samples <- meta_merged$total_counts >= MIN_SAMPLE_COUNTS
  message("\nFiltering: Requiring >= ", MIN_SAMPLE_COUNTS, " counts. ('n_cells' column missing, skipped cell filter).")
}

meta_filtered <- meta_merged[keep_samples, ]
counts_filtered <- counts_merged[, rownames(meta_filtered)]
message("Filtered out ", sum(!keep_samples), " low-quality samples. Kept ", sum(keep_samples), ".")

# 5. Setup factors for DESeq2
meta_filtered$species   <- factor(meta_filtered$species, levels = SPECIES)
meta_filtered$donor     <- factor(meta_filtered$donor)
meta_filtered$age       <- factor(meta_filtered$age)
meta_filtered$region    <- factor(meta_filtered$region)
meta_filtered$is_human  <- factor(ifelse(meta_filtered$species == "Human", "Human", "NonHuman"), levels = c("NonHuman", "Human"))

meta_filtered$cell_type <- as.factor(make.names(as.character(meta_filtered$cell_type)))

# 6. Subset to shared cell types
ct_per_species <- split(as.character(meta_filtered$cell_type), meta_filtered$species)
shared_ct <- Reduce(intersect, ct_per_species)

keep_shared <- meta_filtered$cell_type %in% shared_ct
counts_shared <- counts_filtered[, keep_shared]
meta_shared   <- meta_filtered[keep_shared, ]
meta_shared$cell_type <- droplevels(meta_shared$cell_type)

message("Final subset to shared cell types: ", ncol(counts_shared), " pseudobulk samples ready for DESeq2.")


--- Top & Bottom 3 Samples by Total Counts per Species ---



      species     Rank                                           sample_id
1      Bonobo    Top 1                    Naive B cells_BN114_Adult_Bonobo
2      Bonobo    Top 2                          T cells_BN131_Adult_Bonobo
3      Bonobo    Top 3                         TA cells_BN114_Adult_Bonobo
4      Bonobo Bottom 3 Specialized Fibroblasts RSPO2/3+_BN114_Adult_Bonobo
5      Bonobo Bottom 2                       Adipocytes_BN133_Adult_Bonobo
6      Bonobo Bottom 1                     Enteric glia_BN116_Adult_Bonobo
7  Chimpanzee    Top 1                      T cells_BN083_Adult_Chimpanzee
8  Chimpanzee    Top 2                  Enterocytes_BN083_Adult_Chimpanzee
9  Chimpanzee    Top 3                      T cells_BN085_Adult_Chimpanzee
10 Chimpanzee Bottom 3                 Goblet cells_BN075_Adult_Chimpanzee
11 Chimpanzee Bottom 2            Mesothelial cells_BN083_Adult_Chimpanzee
12 Chimpanzee Bottom 1                    Pericytes_BN075_Adult_Chimpanzee
13    Gorilla    Top 1   

✅ Log-transformed count distribution plot saved.


Filtering: Requiring >= 50000 counts AND >= 100 cells.

Filtered out 319 low-quality samples. Kept 355.

Final subset to shared cell types: 241 pseudobulk samples ready for DESeq2.



In [8]:
# =============================================================================
# Cell 6: Species-Aware Peak Filtering
# =============================================================================
MIN_SAMPLES_PER_SPECIES <- 2  # How many samples within a species need signal?
MIN_SPECIES_ACTIVE <- 6       # How many species must pass the above threshold?

n_before <- nrow(counts_shared)

# 1. Filter: Max count across any sample > 150 (removes total background noise)
keep_count <- apply(counts_shared, 1, max) >= 150

# 2. Filter: Species-aware presence
active_in_species <- sapply(SPECIES, function(sp) {
  sp_cols <- meta_shared$species == sp
  if (sum(sp_cols) > 0) {
    rowSums(counts_shared[, sp_cols, drop = FALSE] >= 10) >= MIN_SAMPLES_PER_SPECIES
  } else {
    rep(FALSE, nrow(counts_shared))
  }
})

# A peak is kept if it is active in at least MIN_SPECIES_ACTIVE
keep_species_thresh <- rowSums(active_in_species) >= MIN_SPECIES_ACTIVE

keep_peaks <- keep_count & keep_species_thresh
counts_shared <- counts_shared[keep_peaks, ]

message("Species-Aware Peak filtering:")
message("  Kept: ", sum(keep_peaks), " (", round(100 * sum(keep_peaks) / n_before, 1), "%)")

# =============================================================================
# Cell 6b: Global VST and Custom PCA (Post-Filtering)
# =============================================================================
message("\nPreparing global data for PCA based on species-aware filtered peaks...")

# Run a global DESeq2 model purely to get the normalized, log-scaled data (VST)
dds_global_filt <- DESeqDataSetFromMatrix(
  countData = round(counts_shared),
  colData   = meta_shared,
  design    = ~ species + cell_type
)

# THE FIX: Apply type="poscounts" here as well!
dds_global_filt <- estimateSizeFactors(dds_global_filt, type = "poscounts")
vsd_global_filt <- vst(dds_global_filt, blind = FALSE)

message("Extracting PCA coordinates...")

# 1. Extract raw PCA data with ALL metadata fields
pca_data <- plotPCA(vsd_global_filt, intgroup = c("cell_type", "species", "donor", "region", "age"), returnData = TRUE)
percentVar <- round(100 * attr(pca_data, "percentVar"))

# 2. Filter and collapse cell types for visualization
# NOTE: Updated to match make.names() formatting from Cell 5!
target_cts <- c("Enterocytes", "Colonocytes", "Goblet.cells", "T.cells", "Naive.B.cells", "Plasma.B.cells")

pca_filtered <- pca_data %>%
  mutate(cell_type_broad = case_when(
    cell_type %in% target_cts ~ as.character(cell_type),
    grepl("Fibroblast", cell_type) ~ "Fibroblasts", 
    TRUE ~ "Drop"
  )) %>%
  filter(cell_type_broad != "Drop")

pca_filtered$cell_type_broad <- factor(pca_filtered$cell_type_broad)

message(sprintf("Plotting PCA for %d selected samples...", nrow(pca_filtered)))

# 3. Base Plotting Function for Conserved Peaks PCA
plot_custom_pca_post <- function(df, color_var, shape_var = NULL, title, add_labels = FALSE) {
  p <- ggplot(df, aes_string(x = "PC1", y = "PC2", color = color_var, shape = shape_var)) +
    geom_point(size = 4, alpha = 0.85) +
    xlab(paste0("PC1: ", percentVar[1], "% variance")) +
    ylab(paste0("PC2: ", percentVar[2], "% variance")) +
    theme_bw() +
    ggtitle(title) +
    theme(
      legend.position = "right",
      legend.title = element_text(face = "bold"),
      plot.title = element_text(face = "bold", size = 14)
    )
  
  # Only add donor labels to the combined plot to prevent total clutter
  if (add_labels) {
    p <- p + ggrepel::geom_text_repel(aes(label = donor), size = 3, show.legend = FALSE, max.overlaps = 20)
  }
  
  # Handle shapes if provided
  if (!is.null(shape_var)) {
    num_shapes <- length(unique(df[[shape_var]]))
    # Using your preferred shape values + some extras just in case
    p <- p + scale_shape_manual(values = c(15, 16, 17, 18, 3, 4, 8, 9)[1:num_shapes]) 
  }
  return(p)
}

# 4. Generate and save all requested PCA variations
pca_plots_post <- list(
  Species   = plot_custom_pca_post(pca_filtered, color_var = "species", title = "Conserved Peaks PCA: Species"),
  CellType  = plot_custom_pca_post(pca_filtered, color_var = "cell_type_broad", title = "Conserved Peaks PCA: Cell Type"),
  Region    = plot_custom_pca_post(pca_filtered, color_var = "region", title = "Conserved Peaks PCA: Region"),
  Age       = plot_custom_pca_post(pca_filtered, color_var = "age", title = "Conserved Peaks PCA: Age"),
  Donor     = plot_custom_pca_post(pca_filtered, color_var = "donor", title = "Conserved Peaks PCA: Donor"),
  Combined  = plot_custom_pca_post(pca_filtered, color_var = "species", shape_var = "cell_type_broad", title = "Conserved Peaks PCA: Species & Cell Type", add_labels = TRUE)
)

plots_dir <- file.path(OUT_DIR, "plots")
dir.create(plots_dir, showWarnings = FALSE, recursive = TRUE)

for (plot_name in names(pca_plots_post)) {
  file_name <- file.path(plots_dir, paste0("Conserved_Peaks_PCA_", plot_name, ".pdf"))
  ggsave(file_name, pca_plots_post[[plot_name]], width = 10, height = 7)
  message("  Saved: ", file_name)
}
message("✅ All Conserved Peak PCA plots generated successfully.")

Species-Aware Peak filtering:

  Kept: 71663 (8.6%)


Preparing global data for PCA based on species-aware filtered peaks...

converting counts to integer mode

Extracting PCA coordinates...

using ntop=500 top features by variance

Plotting PCA for 161 selected samples...

Warning message:
“`aes_string()` was deprecated in ggplot2 3.0.0.
ℹ Please use tidy evaluation idioms with `aes()`.
ℹ See also `vignette("ggplot2-in-packages")` for more information.”
  Saved: /cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/13_deseq2_R_pseudobulk/plots/Conserved_Peaks_PCA_Species.pdf

  Saved: /cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/13_deseq2_R_pseudobulk/plots/Conserved_Peaks_PCA_CellType.pdf

  Saved: /cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/13_deseq2_R_pseudobulk/plots/Conserved_Peaks_PCA_Region.pdf

  Saved: /cluster/project/treutlein/U

In [9]:
# =============================================================================
# Cell 7: Per-Cell-Type DESeq2 (Testing ALL Species)
# =============================================================================
# 1. Setup the main directory to save our results
RES_DIR <- file.path(OUT_DIR, "differential_results")
dir.create(RES_DIR, showWarnings = FALSE, recursive = TRUE)

res_list <- list() 

cell_types <- levels(meta_shared$cell_type)

for (ct in cell_types) {
  message(sprintf("\n=== Processing %s ===", ct))
  
  meta_ct <- meta_shared[meta_shared$cell_type == ct, ]
  counts_ct <- counts_shared[, rownames(meta_ct)]
  
  # Ensure the cell type is actually present in multiple species
  present_species <- unique(as.character(meta_ct$species))
  if (length(present_species) < 2) {
    message("  ⚠️ Skipping: Not enough species represented for this cell type.")
    next
  }
  
  # Local peak filtering for this specific cell type
  keep_ct <- rowSums(counts_ct >= 10) >= 3
  counts_ct_filtered <- counts_ct[keep_ct, ]
  
  # Use ~ 0 + species (removes intercept for clean contrasts)
  dds_ct <- DESeqDataSetFromMatrix(
    countData = round(counts_ct_filtered),
    colData   = meta_ct,
    design    = ~ 0 + species
  )
  
  # ⬇️ THE FIX: We MUST use poscounts here too to avoid the geometric mean error on sparse data
  dds_ct <- estimateSizeFactors(dds_ct, type = "poscounts")
  dds_ct <- DESeq(dds_ct, quiet = TRUE)
  
  res_list[[ct]] <- list()
  
  # Create a subfolder for this cell type's results
  ct_dir <- file.path(RES_DIR, ct)
  dir.create(ct_dir, showWarnings = FALSE)
  
  # Get the exact term names (e.g., "speciesHuman", "speciesMacaque")
  res_names <- resultsNames(dds_ct)
  
  for (target_sp in present_species) {
    
    target_term <- paste0("species", target_sp)
    if (!target_term %in% res_names) next
    
    # Build a numeric contrast vector: Target species = 1, All others = -1 / (N-1)
    contrast_vec <- rep(-1 / (length(res_names) - 1), length(res_names))
    target_idx <- which(res_names == target_term)
    contrast_vec[target_idx] <- 1
    
    # Pass the numeric vector directly to calculate differential accessibility
    res_sp <- results(dds_ct, contrast = contrast_vec)
    res_sp_ordered <- res_sp[order(res_sp$padj), ]
    
    # Save it to our nested list in R memory
    res_list[[ct]][[target_sp]] <- res_sp_ordered
    
    # Format for saving to disk
    res_df <- as.data.frame(res_sp_ordered)
    res_df <- tibble::rownames_to_column(res_df, "region_id") # Keep peak IDs safe
    
    out_file <- file.path(ct_dir, paste0(target_sp, "_vs_All_Others.csv"))
    write.csv(res_df, out_file, row.names = FALSE)
    
    sig_hits <- sum(res_sp$padj < 0.05 & res_sp$log2FoldChange > 1, na.rm = TRUE)
    message(sprintf("  ✅ %-12s specific regions: %d (padj < 0.05, LFC > 1)", target_sp, sig_hits))
  }
}

# Save the full DESeq2 results list so we never have to re-run this!
saveRDS(res_list, file.path(OUT_DIR, "differential_results", "DESeq2_res_list.rds"))
message("💾 Checkpoint: res_list saved.")

message("\n✅ All differential accessibility results successfully saved to: ", RES_DIR)



=== Processing Colonocytes ===

converting counts to integer mode

  ✅ Marmoset     specific regions: 4575 (padj < 0.05, LFC > 1)

  ✅ Human        specific regions: 4856 (padj < 0.05, LFC > 1)

  ✅ Gorilla      specific regions: 716 (padj < 0.05, LFC > 1)

  ✅ Chimpanzee   specific regions: 159 (padj < 0.05, LFC > 1)

  ✅ Bonobo       specific regions: 130 (padj < 0.05, LFC > 1)

  ✅ Macaque      specific regions: 3308 (padj < 0.05, LFC > 1)


=== Processing Crypt.Fibroblasts.WNT2B. ===

converting counts to integer mode

  ✅ Marmoset     specific regions: 3154 (padj < 0.05, LFC > 1)

  ✅ Human        specific regions: 2638 (padj < 0.05, LFC > 1)

  ✅ Gorilla      specific regions: 1100 (padj < 0.05, LFC > 1)

  ✅ Chimpanzee   specific regions: 724 (padj < 0.05, LFC > 1)

  ✅ Bonobo       specific regions: 557 (padj < 0.05, LFC > 1)

  ✅ Macaque      specific regions: 662 (padj < 0.05, LFC > 1)


=== Processing ECs ===

converting counts to integer mode

  ✅ Marmoset     specific reg

In [10]:
# =============================================================================
# Cell 8: Visualization Suite (Heatmap, Volcano, PCA) for ALL Cell Types
# =============================================================================
library(ggplot2)
library(pheatmap)
library(viridis)

# We will iterate only over cell types that successfully ran in Cell 7
processed_cell_types <- names(res_list)

for (target_ct in processed_cell_types) {
  message("\n=== Generating Visualizations for: ", target_ct, " ===")
  
  # Create a tidy subfolder just for this cell type's plots
  ct_plot_dir <- file.path(OUT_DIR, "plots", paste0(target_ct, "_Visualizations"))
  dir.create(ct_plot_dir, showWarnings = FALSE, recursive = TRUE)

  # -----------------------------------------------------------------------------
  # 1. Top Species-Specific Heatmap (All Species)
  # -----------------------------------------------------------------------------
  top_peaks_list <- list()
  
  for (sp in names(res_list[[target_ct]])) {
    res_sp <- res_list[[target_ct]][[sp]]
    # Filter for significance and positive LFC (higher in THIS species)
    sig_res <- res_sp[!is.na(res_sp$padj) & res_sp$padj < 0.05 & res_sp$log2FoldChange > 1, ]
    # Sort by fold change
    sig_res <- sig_res[order(sig_res$log2FoldChange, decreasing = TRUE), ] 
    top_peaks_list[[sp]] <- rownames(head(sig_res, 25))
  }
  
  top_regions <- unique(unlist(top_peaks_list))
  
  if (length(top_regions) > 2) {
    meta_ct <- meta_shared[meta_shared$cell_type == target_ct, ]
    meta_ct <- meta_ct[order(meta_ct$species), ]
    samples_to_plot <- rownames(meta_ct)
    
    # Get log-normalized values from the global VST object
    mat <- assay(vsd_global_filt)[top_regions, samples_to_plot]
    mat_scaled <- t(scale(t(mat))) # Z-score
    
    anno_col <- data.frame(
      Species = meta_ct$species,
      row.names = samples_to_plot
    )
    
    heatmap_file <- file.path(ct_plot_dir, paste0("1_Heatmap_Top_Markers_", target_ct, ".pdf"))
    pheatmap(mat_scaled,
             annotation_col = anno_col,
             cluster_cols = FALSE,  
             cluster_rows = TRUE,   
             show_rownames = FALSE,
             show_colnames = FALSE,
             color = viridis(100), 
             main = paste("Top Species Markers in", target_ct),
             filename = heatmap_file,
             width = 9, height = 7)
    message("  ✅ Heatmap saved!")
  } else {
    message("  ⚠️ Skipping Heatmap: Not enough significant markers found.")
  }

  # -----------------------------------------------------------------------------
  # 2. Volcano & PCA Plots (Human vs All)
  # -----------------------------------------------------------------------------
  # Check if Human was successfully tested for this cell type
  if ("Human" %in% names(res_list[[target_ct]])) {
    res_human <- as.data.frame(res_list[[target_ct]][["Human"]])
    res_human <- res_human[!is.na(res_human$padj), ]
    
    # Label significance thresholds
    res_human$Significance <- "Not Significant"
    res_human$Significance[res_human$padj < 0.05 & res_human$log2FoldChange > 1] <- "Up in Human"
    res_human$Significance[res_human$padj < 0.05 & res_human$log2FoldChange < -1] <- "Down in Human"
    res_human$Significance <- factor(res_human$Significance, levels = c("Up in Human", "Down in Human", "Not Significant"))
    
    volcano_colors <- c("Up in Human" = "#e41a1c", "Down in Human" = "#377eb8", "Not Significant" = "grey80")
    
    volcano_plot <- ggplot(res_human, aes(x = log2FoldChange, y = -log10(padj), color = Significance)) +
      geom_point(alpha = 0.6, size = 1.5) +
      scale_color_manual(values = volcano_colors) +
      geom_vline(xintercept = c(-1, 1), linetype = "dashed", color = "black") +
      geom_hline(yintercept = -log10(0.05), linetype = "dashed", color = "black") +
      theme_bw() +
      labs(
        title = paste("Human vs All Others (", target_ct, ")"),
        x = "Log2 Fold Change",
        y = "-Log10(Adjusted P-Value)"
      ) +
      theme(legend.position = "bottom", legend.title = element_blank())
    
    volcano_file <- file.path(ct_plot_dir, paste0("2_Volcano_Human_", target_ct, ".pdf"))
    ggsave(volcano_file, volcano_plot, width = 7, height = 7)
    message("  ✅ Volcano plot saved!")
    
    # -- PCA Feature Plots --
    top_up_human <- rownames(res_human[res_human$Significance == "Up in Human", ])
    top_up_human <- head(top_up_human[order(res_human[top_up_human, "log2FoldChange"], decreasing = TRUE)], 10)
    
    top_down_human <- rownames(res_human[res_human$Significance == "Down in Human", ])
    top_down_human <- head(top_down_human[order(res_human[top_down_human, "log2FoldChange"], decreasing = FALSE)], 10)
    
    top_20_human <- c(top_up_human, top_down_human)
    
    if (length(top_20_human) > 0) {
      pca_base <- plotPCA(vsd_global_filt, intgroup = c("species", "cell_type"), returnData = TRUE)
      percentVar <- round(100 * attr(pca_base, "percentVar"))
      
      pca_feature_file <- file.path(ct_plot_dir, paste0("3_PCA_Features_Top20_Human_", target_ct, ".pdf"))
      pdf(pca_feature_file, width = 10, height = 7)
      
      for (region_id in top_20_human) {
        if (region_id %in% rownames(vsd_global_filt)) {
          pca_base$Region_Accessibility <- assay(vsd_global_filt)[region_id, rownames(pca_base)]
          direction <- ifelse(region_id %in% top_up_human, "(UP in Human)", "(DOWN in Human)")
          
          p <- ggplot(pca_base, aes(x = PC1, y = PC2, color = Region_Accessibility, shape = species)) +
            geom_point(size = 4, alpha = 0.85) +
            scale_color_viridis_c(option = "magma") + 
            scale_shape_manual(values = c(15, 16, 17, 18, 3, 4, 8)[1:length(unique(pca_base$species))]) +
            xlab(paste0("PC1: ", percentVar[1], "% variance")) +
            ylab(paste0("PC2: ", percentVar[2], "% variance")) +
            theme_bw() +
            ggtitle(paste("Accessibility of", region_id, direction), 
                    subtitle = paste("Targeted Cell Type:", target_ct)) +
            theme(legend.position = "right", plot.title = element_text(face = "bold"))
          
          print(p)
        }
      }
      dev.off()
      message("  ✅ Multi-page PCA Feature Plot saved!")
    } else {
      message("  ⚠️ Skipping PCA Features: No significant Human up/down markers found.")
    }
    
  } else {
    message("  ⚠️ Skipping Human Volcano & PCA: 'Human' contrast not available for ", target_ct)
  }
}

message("\n🎉 All visualization suites successfully completed!")

Loading required package: viridisLite


=== Generating Visualizations for: Colonocytes ===

  ✅ Heatmap saved!

  ✅ Volcano plot saved!

using ntop=500 top features by variance

  ✅ Multi-page PCA Feature Plot saved!


=== Generating Visualizations for: Crypt.Fibroblasts.WNT2B. ===

  ✅ Heatmap saved!

  ✅ Volcano plot saved!

using ntop=500 top features by variance

  ✅ Multi-page PCA Feature Plot saved!


=== Generating Visualizations for: ECs ===

  ✅ Heatmap saved!

  ✅ Volcano plot saved!

using ntop=500 top features by variance

  ✅ Multi-page PCA Feature Plot saved!


=== Generating Visualizations for: Enterocytes ===

  ✅ Heatmap saved!

  ✅ Volcano plot saved!

using ntop=500 top features by variance

  ✅ Multi-page PCA Feature Plot saved!


=== Generating Visualizations for: Goblet.cells ===

  ✅ Heatmap saved!

  ✅ Volcano plot saved!

using ntop=500 top features by variance

  ✅ Multi-page PCA Feature Plot saved!


=== Generating Visualizations for: Macrophages ===

  ✅ Hea

In [11]:
# =============================================================================
# Cell 9: Cross-Cell-Type Human Specificity Heatmap (Signed P-Value)
# =============================================================================
library(pheatmap)

message("\n=== Generating Cross-Cell-Type Signed P-Value Heatmap ===")

signed_p_list <- list()
sig_regions_all <- c()

# 1. Iterate over all cell types and extract Human statistics
for (ct in names(res_list)) {
  if ("Human" %in% names(res_list[[ct]])) {
    res_df <- as.data.frame(res_list[[ct]][["Human"]])
    
    # Define significant regions for this cell type (Padj < 0.05 & |LFC| > 1)
    sig_idx <- which(!is.na(res_df$padj) & res_df$padj < 0.05 & abs(res_df$log2FoldChange) > 1)
    sig_regions_all <- c(sig_regions_all, rownames(res_df)[sig_idx])
    
    # Handle NAs and absolute 0s in p-values to prevent mathematical errors (Inf)
    padj_safe <- res_df$padj
    padj_safe[is.na(padj_safe)] <- 1       # -log10(1) = 0 (No signal)
    padj_safe[padj_safe == 0] <- 1e-300    # Cap tiny p-values to prevent Infinity
    
    # Calculate the metric: LogFoldChange * -log10(P-value)
    # Positive = Up in Human, highly significant. Negative = Down in Human, highly significant.
    signed_pval <- res_df$log2FoldChange * -log10(padj_safe)
    
    # Store in a temporary dataframe
    df <- data.frame(region = rownames(res_df), metric = signed_pval)
    colnames(df)[2] <- ct # Name the column after the cell type
    signed_p_list[[ct]] <- df
  }
}

# 2. Get the unique list of regions that were significant in AT LEAST ONE cell type
sig_regions_all <- unique(sig_regions_all)
message("Found ", length(sig_regions_all), " unique regions significant in at least one cell type.")

if (length(sig_regions_all) > 0) {
  # 3. Merge all cell type metrics into a single master matrix
  message("Merging cell type results...")
  merged_df <- Reduce(function(x, y) merge(x, y, by = "region", all = TRUE), signed_p_list)
  rownames(merged_df) <- merged_df$region
  merged_df$region <- NULL
  
  # 4. Subset to only our significant regions and convert to matrix
  plot_mat <- as.matrix(merged_df[sig_regions_all, ])
  
  # Any NAs left mean that region was filtered out by DESeq2 in that specific cell type
  # We set those to 0 (no significant signal)
  plot_mat[is.na(plot_mat)] <- 0
  
  # 5. Cap extreme values for better visualization (99th percentile)
  # This prevents 1 or 2 mega-peaks from turning the rest of the heatmap grey
  cap_val <- quantile(abs(plot_mat), 0.99, na.rm = TRUE)
  plot_mat[plot_mat > cap_val] <- cap_val
  plot_mat[plot_mat < -cap_val] <- -cap_val
  
  # 6. Set up the custom color palette
  # Deep Blue (Down in Human) -> White (Neutral) -> Deep Red (Up in Human)
  bwr_palette <- colorRampPalette(c("#377eb8", "white", "#e41a1c"))(100)
  
  # Center the color breaks strictly around 0
  max_abs <- max(abs(plot_mat))
  breaks <- seq(-max_abs, max_abs, length.out = 101)
  
  # 7. Draw the Heatmap
  out_file <- file.path(OUT_DIR, "plots", "Cross_CellType_Human_Specificity_Heatmap.pdf")
  
  pheatmap(plot_mat,
           color = bwr_palette,
           breaks = breaks,
           cluster_cols = TRUE,   # Groups cell types with similar human-specific profiles together
           cluster_rows = TRUE,   # Groups peaks by their specificity patterns (e.g. universal vs specific)
           show_rownames = FALSE, # Too many regions to print names
           show_colnames = TRUE,
           main = "Human-Specific Regions Across All Cell Types\n(Log2FC * -log10(padj))",
           filename = out_file,
           width = 10, height = 8)
  
  message("✅ Cross-cell-type specificity heatmap saved to: ", out_file)
  
} else {
  message("⚠️ No significant human-specific regions found across any cell types. Skipping heatmap.")
}

# Save the master matrix used for clustering
saveRDS(plot_mat, file.path(OUT_DIR, "differential_results", "Signed_Pval_Matrix.rds"))
message("💾 Checkpoint: plot_mat saved.")


=== Generating Cross-Cell-Type Signed P-Value Heatmap ===

Found 27131 unique regions significant in at least one cell type.

Merging cell type results...

✅ Cross-cell-type specificity heatmap saved to: /cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/13_deseq2_R_pseudobulk/plots/Cross_CellType_Human_Specificity_Heatmap.pdf

💾 Checkpoint: plot_mat saved.



In [15]:
# =============================================================================
# Cell 10: K-Means Module Extraction & Clean ComplexHeatmap (All Modules)
# =============================================================================
suppressPackageStartupMessages({
  library(ComplexHeatmap)
  library(circlize)
})

message("\n=== Extracting K-Means Heatmap Modules ===")

NUM_MODULES <- 40         # Your chosen number of modules
MIN_REGIONS <- 5          

# 1. Run K-means clustering
set.seed(42)
km_res <- kmeans(plot_mat, centers = NUM_MODULES, iter.max = 100, nstart = 25)

# 2. Extract assignments and sizes
cluster_assignments <- km_res$cluster
module_sizes <- table(cluster_assignments)

keep_modules <- as.numeric(names(module_sizes[module_sizes >= MIN_REGIONS]))
message(sprintf("Filtering: Keeping %d out of %d modules (>= %d regions).", 
                length(keep_modules), NUM_MODULES, MIN_REGIONS))

# 3. Analyze Behavior & Sort Data for Smooth Plotting
cluster_dir <- file.path(OUT_DIR, "differential_results", "cross_celltype_modules")
dir.create(cluster_dir, showWarnings = FALSE, recursive = TRUE)

row_order <- c()
ordered_assignments <- c()
csv_behavior_list <- list()

for (mod_num in keep_modules) {
  mod_regions <- names(cluster_assignments)[cluster_assignments == mod_num]
  
  # Determine biological behavior
  center_vals <- km_res$centers[mod_num, ]
  top_ct <- names(center_vals)[which.max(center_vals)]
  max_val <- max(center_vals)
  min_val <- min(center_vals)
  
  if (min_val > (max_val * 0.5) && max_val > 0) {
    behavior <- "Universal (Up)"
  } else if (max_val < 0) {
    behavior <- "Universal (Down)"
  } else {
    behavior <- paste("Strong in", top_ct)
  }
  
  csv_behavior_list[[as.character(mod_num)]] <- behavior
  message(sprintf("  ✅ Module %d: %-25s (%d regions)", mod_num, behavior, length(mod_regions)))
  
  # Internal sorting: Sort regions in this module by maximum absolute signal
  mod_mat <- plot_mat[mod_regions, , drop = FALSE]
  internal_order <- order(apply(abs(mod_mat), 1, max), decreasing = TRUE)
  sorted_mod_regions <- mod_regions[internal_order]
  
  row_order <- c(row_order, sorted_mod_regions)
  ordered_assignments <- c(ordered_assignments, rep(mod_num, length(sorted_mod_regions)))
  
  # Save to CSV
  out_file <- file.path(cluster_dir, paste0("Module_", mod_num, "_regions.csv"))
  write.csv(data.frame(region_id = sorted_mod_regions), out_file, row.names = FALSE)
}

# Save master mapping for your records
master_df <- data.frame(
  region_id = row_order,
  module = paste0("Module_", ordered_assignments),
  behavior = sapply(as.character(ordered_assignments), function(x) csv_behavior_list[[x]])
)
write.csv(master_df, file.path(cluster_dir, "All_Regions_Module_Assignments.csv"), row.names = FALSE)

# 4. Filter and Apply our Custom Row Order
plot_mat_ordered <- plot_mat[row_order, , drop = FALSE]


# 5. Draw the Clean ComplexHeatmap
message("\nDrawing ComplexHeatmap...")

split_factor <- factor(paste0("Module ", ordered_assignments), 
                       levels = paste0("Module ", keep_modules))

# Define the color mapping cleanly
max_abs <- max(abs(plot_mat_ordered))
col_fun <- colorRamp2(c(-max_abs, 0, max_abs), c("#377eb8", "white", "#e41a1c"))

# 👈 Dynamically adjust font size for labels if there are a ton of modules
label_size <- ifelse(length(keep_modules) > 20, 8, 12)

ht <- Heatmap(plot_mat_ordered,
              name = "Signed_Pval",             
              col = col_fun,
              
              # Row Splitting & Minimal Labeling
              row_split = split_factor,
              row_title_rot = 0,                
              row_title_gp = gpar(fontsize = label_size, fontface = "bold"),
              row_gap = unit(2, "mm"), # 👈 Reduced gap size so 40 blocks fit better!        
              
              # Disable Row Clustering! 
              cluster_rows = FALSE,              
              show_row_names = FALSE,
              
              # Column Labeling Formatting
              cluster_columns = TRUE,           
              column_names_rot = 45,            
              column_names_gp = gpar(fontsize = 11),
              
              # Aesthetics
              border = TRUE,                    
              column_title = paste("Human-Specific K-Means Modules (k =", length(keep_modules), ")"),
              column_title_gp = gpar(fontsize = 14, fontface = "bold"),
              heatmap_legend_param = list(title = "Signed\n-log10(padj)") 
)

annotated_heatmap_file <- file.path(OUT_DIR, "plots", "Cross_CellType_ComplexHeatmap_Clean.pdf")

# 👈 Dynamically scale PDF height based on number of modules
dynamic_height <- max(9, length(keep_modules) * 0.4) 

pdf(annotated_heatmap_file, width = 11, height = dynamic_height)
draw(ht, merge_legend = TRUE)
dev.off()

message("✅ Clean ComplexHeatmap saved to: ", annotated_heatmap_file)


=== Extracting K-Means Heatmap Modules ===

Filtering: Keeping 40 out of 40 modules (>= 5 regions).

  ✅ Module 1: Strong in Enterocytes     (1727 regions)

  ✅ Module 2: Strong in Enterocytes     (410 regions)

  ✅ Module 3: Strong in ECs             (1462 regions)

  ✅ Module 4: Universal (Down)          (138 regions)

  ✅ Module 5: Universal (Down)          (89 regions)

  ✅ Module 6: Strong in Goblet.cells    (915 regions)

  ✅ Module 7: Universal (Down)          (202 regions)

  ✅ Module 8: Universal (Down)          (1812 regions)

  ✅ Module 9: Strong in TA.cells        (180 regions)

  ✅ Module 10: Universal (Down)          (591 regions)

  ✅ Module 11: Strong in Plasma.B.cells  (963 regions)

  ✅ Module 12: Strong in ECs             (215 regions)

  ✅ Module 13: Universal (Down)          (375 regions)

  ✅ Module 14: Strong in Colonocytes     (146 regions)

  ✅ Module 15: Strong in Crypt.Fibroblasts.WNT2B. (438 regions)

  ✅ Module 16: Universal (Down)          (1671 regions)


pdf 
  2

✅ Clean ComplexHeatmap saved to: /cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/13_deseq2_R_pseudobulk/plots/Cross_CellType_ComplexHeatmap_Clean.pdf



In [16]:
# =============================================================================
# Cell 11: Convert Peaks to BED Coordinates (Cell Types & Modules)
# =============================================================================
library(readr)

message("\n=== Converting Peak IDs to BED Coordinates ===")

# 1. Define the Coordinate Lookup Function
get_coordinates <- function(peak_ids, species, anno_df) {
  if (length(peak_ids) == 0) return(NULL)
  
  chr_col   <- paste0(species, "_chr")
  start_col <- paste0(species, "_start")
  end_col   <- paste0(species, "_end")
  
  if (!all(c(chr_col, start_col, end_col) %in% colnames(anno_df))) {
    stop("Coordinates for ", species, " not found in annotation dataframe.")
  }
  
  # Subset to matching peaks
  sub_anno <- anno_df[anno_df$peak_id %in% peak_ids, c("peak_id", chr_col, start_col, end_col)]
  
  # Filter out regions that don't actually exist in this species (NAs or ".")
  sub_anno <- sub_anno[!is.na(sub_anno[[chr_col]]) & sub_anno[[chr_col]] != ".", ]
  
  if (nrow(sub_anno) == 0) return(NULL)
  
  # Standard BED format: chrom, chromStart, chromEnd, name
  bed_df <- data.frame(
    chrom      = sub_anno[[chr_col]],
    chromStart = as.integer(sub_anno[[start_col]]), 
    chromEnd   = as.integer(sub_anno[[end_col]]),
    name       = sub_anno$peak_id
  )
  
  return(bed_df)
}

# 2. Load the Master Annotation
anno_file <- file.path(BASE, "cross_species_consensus_v3/07_master_annotation/master_annotation_final.tsv")
message("Loading master annotation from: ", anno_file)
master_anno <- read_tsv(anno_file, show_col_types = FALSE)

# Disable scientific notation globally so coordinates write properly (e.g., 1000000 not 1e+06)
options(scipen = 999) 


# -----------------------------------------------------------------------------
# Part A: Generate BED files per Cell Type (Up and Down)
# -----------------------------------------------------------------------------
message("\n--- Processing Cell Types ---")
RES_DIR <- file.path(OUT_DIR, "differential_results")

for (ct in names(res_list)) {
  if ("Human" %in% names(res_list[[ct]])) {
    res_df <- as.data.frame(res_list[[ct]][["Human"]])
    ct_dir <- file.path(RES_DIR, ct)
    
    # Isolate UP and DOWN peaks
    up_peaks   <- rownames(res_df)[!is.na(res_df$padj) & res_df$padj < 0.05 & res_df$log2FoldChange > 1]
    down_peaks <- rownames(res_df)[!is.na(res_df$padj) & res_df$padj < 0.05 & res_df$log2FoldChange < -1]
    
    # Convert and Save UP
    up_bed <- get_coordinates(up_peaks, "Human", master_anno)
    if (!is.null(up_bed)) {
      out_up <- file.path(ct_dir, paste0("Human_UP_Regions.bed"))
      write.table(up_bed, out_up, sep = "\t", quote = FALSE, row.names = FALSE, col.names = FALSE)
      message(sprintf("  ✅ %s: Saved %d UP regions", ct, nrow(up_bed)))
    }
    
    # Convert and Save DOWN
    down_bed <- get_coordinates(down_peaks, "Human", master_anno)
    if (!is.null(down_bed)) {
      out_down <- file.path(ct_dir, paste0("Human_DOWN_Regions.bed"))
      write.table(down_bed, out_down, sep = "\t", quote = FALSE, row.names = FALSE, col.names = FALSE)
      message(sprintf("  ✅ %s: Saved %d DOWN regions", ct, nrow(down_bed)))
    }
  }
}


# -----------------------------------------------------------------------------
# Part B: Generate BED files per K-Means Module
# -----------------------------------------------------------------------------
message("\n--- Processing K-Means Modules ---")
cluster_dir <- file.path(OUT_DIR, "differential_results", "cross_celltype_modules")
module_file <- file.path(cluster_dir, "All_Regions_Module_Assignments.csv")

if (file.exists(module_file)) {
  mod_df <- read.csv(module_file)
  modules <- unique(mod_df$module)
  
  for (mod in modules) {
    mod_peaks <- mod_df$region_id[mod_df$module == mod]
    
    mod_bed <- get_coordinates(mod_peaks, "Human", master_anno)
    
    if (!is.null(mod_bed)) {
      out_mod <- file.path(cluster_dir, paste0(mod, "_regions.bed"))
      write.table(mod_bed, out_mod, sep = "\t", quote = FALSE, row.names = FALSE, col.names = FALSE)
      message(sprintf("  ✅ %s: Saved %d regions", mod, nrow(mod_bed)))
    }
  }
} else {
  message("  ⚠️ Module assignments CSV not found. Did you run Cell 10?")
}

message("\n🎉 All BED conversions complete!")


=== Converting Peak IDs to BED Coordinates ===

Loading master annotation from: /cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/07_master_annotation/master_annotation_final.tsv


--- Processing Cell Types ---

  ✅ Colonocytes: Saved 4856 UP regions

  ✅ Colonocytes: Saved 1721 DOWN regions

  ✅ Crypt.Fibroblasts.WNT2B.: Saved 2638 UP regions

  ✅ Crypt.Fibroblasts.WNT2B.: Saved 1518 DOWN regions

  ✅ ECs: Saved 171 UP regions

  ✅ ECs: Saved 80 DOWN regions

  ✅ Enterocytes: Saved 5624 UP regions

  ✅ Enterocytes: Saved 2365 DOWN regions

  ✅ Goblet.cells: Saved 5290 UP regions

  ✅ Goblet.cells: Saved 2821 DOWN regions

  ✅ Macrophages: Saved 1761 UP regions

  ✅ Macrophages: Saved 616 DOWN regions

  ✅ Naive.B.cells: Saved 806 UP regions

  ✅ Naive.B.cells: Saved 226 DOWN regions

  ✅ Plasma.B.cells: Saved 3143 UP regions

  ✅ Plasma.B.cells: Saved 1335 DOWN regions

  ✅ Specialized.Fibroblasts.RSPO3..only: Saved 956 UP regions

  ✅ S

In [17]:
# =============================================================================
# Cell 11: Convert Peaks to BED Coordinates using Master Annotation
# =============================================================================
library(readr)

message("\n=== Converting Peak IDs to BED Coordinates ===")

# 1. Define the Coordinate Lookup Function
get_coordinates <- function(peak_ids, species, anno_df) {
  # Build column names dynamically based on target species
  chr_col   <- paste0(species, "_chr")
  start_col <- paste0(species, "_start")
  end_col   <- paste0(species, "_end")
  
  # Check if columns exist
  if (!all(c(chr_col, start_col, end_col) %in% colnames(anno_df))) {
    stop("Coordinates for ", species, " not found in annotation dataframe.")
  }
  
  # Subset to matching peaks and grab coordinate columns
  sub_anno <- anno_df[anno_df$peak_id %in% peak_ids, c("peak_id", chr_col, start_col, end_col)]
  
  # Filter out regions that don't actually exist in this species (NAs)
  sub_anno <- sub_anno[!is.na(sub_anno[[chr_col]]) & sub_anno[[chr_col]] != ".", ]
  
  # Standard BED format: chrom, chromStart, chromEnd, name
  bed_df <- data.frame(
    chrom      = sub_anno[[chr_col]],
    # Convert to integer to prevent R from printing scientific notation (e.g. 1e6)
    chromStart = as.integer(sub_anno[[start_col]]), 
    chromEnd   = as.integer(sub_anno[[end_col]]),
    name       = sub_anno$peak_id
  )
  
  return(bed_df)
}

# 2. Load the Master Annotation
anno_file <- file.path(BASE, "cross_species_consensus_v3/07_master_annotation/master_annotation_final.tsv")
message("Loading master annotation from: ", anno_file)
master_anno <- read_tsv(anno_file, show_col_types = FALSE)

# 3. Gather all Human-Specific (UP) peaks across all cell types
human_up_peaks <- c()

for (ct in names(res_list)) {
  if ("Human" %in% names(res_list[[ct]])) {
    res_df <- as.data.frame(res_list[[ct]][["Human"]])
    
    # Significant and UP in Human
    sig_up <- rownames(res_df)[!is.na(res_df$padj) & res_df$padj < 0.05 & res_df$log2FoldChange > 1]
    human_up_peaks <- c(human_up_peaks, sig_up)
  }
}

human_up_peaks <- unique(human_up_peaks)
message("Found ", length(human_up_peaks), " unique regions significantly UP in Human across all cell types.")

# 4. Apply the function to get Human coordinates
human_bed <- get_coordinates(peak_ids = human_up_peaks, species = "Human", anno_df = master_anno)
message("Successfully mapped ", nrow(human_bed), " regions to Human coordinates.")

# 5. Save the BED file
# Disable scientific notation globally just to be safe for BED coordinate writing
options(scipen = 999) 

bed_dir <- file.path(OUT_DIR, "differential_results", "bed_files")
dir.create(bed_dir, showWarnings = FALSE, recursive = TRUE)

out_bed_file <- file.path(bed_dir, "All_Human_Specific_UP_Regions.bed")

# write.table is preferred for BED files to strictly remove quotes and headers
write.table(human_bed, out_bed_file, 
            sep = "\t", quote = FALSE, row.names = FALSE, col.names = FALSE)

message("✅ BED file saved to: ", out_bed_file)


=== Converting Peak IDs to BED Coordinates ===

Loading master annotation from: /cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/07_master_annotation/master_annotation_final.tsv

Found 17955 unique regions significantly UP in Human across all cell types.

Successfully mapped 17955 regions to Human coordinates.

✅ BED file saved to: /cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/13_deseq2_R_pseudobulk/differential_results/bed_files/All_Human_Specific_UP_Regions.bed



In [18]:
# =============================================================================
# Updated Swiss-Army-Knife Lookup Function
# =============================================================================
get_peak_info <- function(peak_ids, species, info_type = "coordinates", anno_df = master_anno) {
  # Valid info_types: "coordinates", "gene", "distance", "status"
  
  # 1. Coordinate columns
  chr_col   <- paste0(species, "_chr")
  start_col <- paste0(species, "_start")
  end_col   <- paste0(species, "_end")
  
  # 2. Gene/Distance columns
  gene_col  <- paste0(species, "_gene")
  dist_col  <- paste0(species, "_gene_dist")
  
  # 3. Status columns
  det_col   <- paste0(species, "_det")
  orth_col  <- paste0(species, "_orth")
  
  # Filter annotation for target peaks
  sub_anno <- anno_df[anno_df$peak_id %in% peak_ids, ]
  
  if (info_type == "coordinates") {
    res <- sub_anno[!is.na(sub_anno[[chr_col]]) & sub_anno[[chr_col]] != ".", ]
    return(data.frame(chrom = res[[chr_col]], start = as.integer(res[[start_col]]), 
                      end = as.integer(res[[end_col]]), name = res$peak_id))
    
  } else if (info_type == "gene") {
    return(data.frame(peak_id = sub_anno$peak_id, gene = sub_anno[[gene_col]]))
    
  } else if (info_type == "distance") {
    return(data.frame(peak_id = sub_anno$peak_id, distance = sub_anno[[dist_col]]))
    
  } else if (info_type == "status") {
    # Returns 1/1 (Det/Orth), 0/1 (Not Det/Orth), etc.
    return(data.frame(peak_id = sub_anno$peak_id, 
                      status = paste0(sub_anno[[det_col]], "/", sub_anno[[orth_col]])))
  }
}

In [19]:
# =============================================================================
# Cell 12: GO Enrichment Analysis per Module
# =============================================================================
library(clusterProfiler)
library(org.Hs.eg.db)

# Folder to save enrichment results
enrich_dir <- file.path(OUT_DIR, "differential_results", "enrichment")
dir.create(enrich_dir, showWarnings = FALSE, recursive = TRUE)

# We will test the Top Modules you identified in Cell 10
# (Change 'keep_modules' or 'NUM_MODULES' if you want a different subset)
for (mod_num in keep_modules) {
  message("Running enrichment for Module ", mod_num, "...")
  
  # 1. Get Peak IDs for this module
  mod_peaks <- master_df$region_id[master_df$module == paste0("Module_", mod_num)]
  
  # 2. Get associated Human Genes
  gene_info <- get_peak_info(mod_peaks, "Human", info_type = "gene")
  mod_genes <- unique(gene_info$gene)
  mod_genes <- mod_genes[mod_genes != "." & !is.na(mod_genes)]
  
  if (length(mod_genes) < 5) {
    message("  ⚠️ Too few genes in Module ", mod_num, " for enrichment.")
    next
  }
  
  # 3. Perform GO Enrichment (Biological Process)
  ego <- enrichGO(gene          = mod_genes,
                  OrgDb         = org.Hs.eg.db,
                  keyType       = 'SYMBOL',
                  ont           = "BP", 
                  pAdjustMethod = "BH",
                  pvalueCutoff  = 0.05,
                  qvalueCutoff  = 0.2)
  
  if (!is.null(ego) && nrow(ego) > 0) {
    # Save CSV results
    write.csv(as.data.frame(ego), 
              file.path(enrich_dir, paste0("Module_", mod_num, "_GO_Enrichment.csv")))
    
    # Save a Dotplot of the top 15 terms
    p <- dotplot(ego, showCategory = 15) + 
         ggtitle(paste("Module", mod_num, "Enrichment"))
    
    ggsave(file.path(enrich_dir, paste0("Module_", mod_num, "_GO_dotplot.pdf")), 
           p, width = 8, height = 7)
    
    message("  ✅ Saved enrichment for Module ", mod_num)
  } else {
    message("  ⚠️ No significant GO terms found for Module ", mod_num)
  }
}



clusterProfiler v4.14.0 Learn more at https://yulab-smu.top/contribution-knowledge-mining/

Please cite:

T Wu, E Hu, S Xu, M Chen, P Guo, Z Dai, T Feng, L Zhou, W Tang, L Zhan,
X Fu, S Liu, X Bo, and G Yu. clusterProfiler 4.0: A universal
enrichment tool for interpreting omics data. The Innovation. 2021,
2(3):100141


Attaching package: ‘clusterProfiler’


The following object is masked from ‘package:IRanges’:

    slice


The following object is masked from ‘package:S4Vectors’:

    rename


The following object is masked from ‘package:stats’:

    filter


Loading required package: AnnotationDbi


Attaching package: ‘AnnotationDbi’


The following object is masked from ‘package:clusterProfiler’:

    select


The following object is masked from ‘package:dplyr’:

    select




Running enrichment for Module 1...

  ⚠️ No significant GO terms found for Module 1

Running enrichment for Module 2...

  ⚠️ No significant GO terms found for Module 2

Running enrichment for Module 3...

 

In [20]:
# =============================================================================
# Updated Swiss-Army-Knife Lookup Function
# =============================================================================
get_peak_info <- function(peak_ids, species, info_type = "coordinates", anno_df = master_anno) {
  # Column mappings
  chr_col   <- paste0(species, "_chr"); start_col <- paste0(species, "_start"); end_col <- paste0(species, "_end")
  gene_col  <- paste0(species, "_gene"); dist_col  <- paste0(species, "_gene_dist")
  det_col   <- paste0(species, "_det"); orth_col  <- paste0(species, "_orth")
  
  sub_anno <- anno_df[anno_df$peak_id %in% peak_ids, ]
  
  if (info_type == "coordinates") {
    res <- sub_anno[!is.na(sub_anno[[chr_col]]) & sub_anno[[chr_col]] != ".", ]
    return(data.frame(seqnames = res[[chr_col]], start = as.integer(res[[start_col]]), 
                      end = as.integer(res[[end_col]]), name = res$peak_id))
  } else if (info_type == "gene") {
    return(data.frame(peak_id = sub_anno$peak_id, gene = sub_anno[[gene_col]]))
  } else if (info_type == "status") {
    return(data.frame(peak_id = sub_anno$peak_id, 
                      status = paste0(sub_anno[[det_col]], "/", sub_anno[[orth_col]])))
  }
}

In [21]:
# =============================================================================
# Cell 12: Functional Enrichment (GREAT & GSEA)
# =============================================================================
library(rGREAT)
library(clusterProfiler)
library(org.Hs.eg.db)

enrich_dir <- file.path(OUT_DIR, "differential_results", "enrichment")
dir.create(enrich_dir, showWarnings = FALSE, recursive = TRUE)

# -----------------------------------------------------------------------------
# 1. GREAT Analysis (Using rGREAT)
# -----------------------------------------------------------------------------
# We'll run GREAT on a few representative modules (e.g., the first 5)
for (mod_num in head(keep_modules, 5)) {
  message("\n--- Running GREAT for Module ", mod_num, " ---")
  
  mod_peaks <- master_df$region_id[master_df$module == paste0("Module_", mod_num)]
  gr_coords <- get_peak_info(mod_peaks, "Human", info_type = "coordinates")
  
  # Submit to GREAT (using hg38 as default)
  job <- submitGreatJob(gr_coords, species = "hg38")
  res_great <- getEnrichmentTables(job)
  
  # Save the GO Biological Process results from GREAT
  if ("GO Biological Process" %in% names(res_great)) {
    write.csv(res_great[["GO Biological Process"]], 
              file.path(enrich_dir, paste0("Module_", mod_num, "_GREAT_GO_BP.csv")))
    message("  ✅ GREAT results saved.")
  }
}

# -----------------------------------------------------------------------------
# 2. GSEA (Gene Set Enrichment Analysis)
# -----------------------------------------------------------------------------
# We run GSEA per cell type using the ranked signed p-values
for (ct in names(res_list)) {
  if (!"Human" %in% names(res_list[[ct]])) next
  
  message("\n--- Running GSEA for Cell Type: ", ct, " ---")
  
  # Get the signed p-values we calculated for the heatmap
  # We need to map peaks to genes for GSEA
  res_human <- as.data.frame(res_list[[ct]][["Human"]])
  res_human$peak_id <- rownames(res_human)
  
  # Join with annotation to get genes
  gene_map <- master_anno[, c("peak_id", "Human_gene")]
  res_human <- left_join(res_human, gene_map, by = "peak_id")
  
  # Calculate rank metric: sign(LFC) * -log10(pval)
  res_human$stat <- res_human$log2FoldChange * -log10(ifelse(res_human$pvalue == 0, 1e-300, res_human$pvalue))
  
  # Collapse multiple peaks per gene by taking the one with the highest absolute rank
  gene_list <- res_human %>%
    filter(!is.na(Human_gene) & Human_gene != ".") %>%
    group_by(Human_gene) %>%
    slice_max(order_by = abs(stat), n = 1) %>%
    pull(stat, name = Human_gene)
  
  # Sort in decreasing order
  gene_list <- sort(gene_list, decreasing = TRUE)
  
  # Run GSEA
  gsea_res <- gseGO(geneList     = gene_list,
                    OrgDb        = org.Hs.eg.db,
                    keyType      = "SYMBOL",
                    ont          = "BP",
                    minGSSize    = 10,
                    maxGSSize    = 500,
                    pvalueCutoff = 0.05,
                    verbose      = FALSE)
  
  if (!is.null(gsea_res) && nrow(gsea_res) > 0) {
    write.csv(as.data.frame(gsea_res), file.path(enrich_dir, paste0(ct, "_GSEA_GO_BP.csv")))
    
    # Plot top pathways
    p <- dotplot(gsea_res, showCategory = 10, split = ".sign") + facet_grid(.~.sign)
    ggsave(file.path(enrich_dir, paste0(ct, "_GSEA_dotplot.pdf")), p, width = 10, height = 8)
    message("  ✅ GSEA results and dotplot saved.")
  }
}

rGREAT version 2.8.0
Bioconductor page: http://bioconductor.org/packages/rGREAT/
Github page: https://github.com/jokergoo/rGREAT

If you use it in published research, please cite:
Gu, Z. rGREAT: an R/Bioconductor package for functional enrichment 
  on genomic regions. Bioinformatics 2023.

This message can be suppressed by:
  suppressPackageStartupMessages(library(rGREAT))

Note: On Aug 19 2019 GREAT released version 4 where it supports `hg38`
genome and removes some ontologies such pathways. `submitGreatJob()`
still takes `hg19` as default. `hg38` can be specified by the `genome
= 'hg38'` argument. To use the older versions such as 3.0.0, specify as
`submitGreatJob(..., version = '3.0.0')`.

From rGREAT version 1.99.0, it also implements the GREAT algorithm and
it allows to integrate GREAT analysis with the Bioconductor annotation
ecosystem. By default it supports more than 500 organisms. Check the
new function `great()` and the new vignette.



--- Running GREAT for Module 1 ---

No

  |===================================================================== |  99%


The default enrichment table does not contain informatin of associated
genes for each input region. You can set `download_by = 'tsv'` to
download the complete table, but note only the top 500 regions can be
retreived. See the following link:

https://great-help.atlassian.net/wiki/spaces/GREAT/pages/655401/Export#Export-GlobalExport

Except the additional gene-region association column if taking 'tsv' as
the source of result, all other columns are the same if you choose
'json' (the default) as the source. Or you can try the local GREAT
analysis with the function `great()`.

  ✅ GREAT results saved.


--- Running GREAT for Module 3 ---

Note: On Aug 19 2019 GREAT released version 4 which supports hg38
genome and removes some ontologies such pathways. submitGreatJob()
still takes hg19 as default. hg38 can be specified by argument `genome
= "hg38"`. To use the older versions such as 3.0.0, specify as
submitGreatJob(..., version = "3"). Set argument `help` to `FALSE` to
turn off this messag

  |======================================================================| 100%


The default enrichment table does not contain informatin of associated
genes for each input region. You can set `download_by = 'tsv'` to
download the complete table, but note only the top 500 regions can be
retreived. See the following link:

https://great-help.atlassian.net/wiki/spaces/GREAT/pages/655401/Export#Export-GlobalExport

Except the additional gene-region association column if taking 'tsv' as
the source of result, all other columns are the same if you choose
'json' (the default) as the source. Or you can try the local GREAT
analysis with the function `great()`.

  ✅ GREAT results saved.


--- Running GREAT for Module 4 ---

Note: On Aug 19 2019 GREAT released version 4 which supports hg38
genome and removes some ontologies such pathways. submitGreatJob()
still takes hg19 as default. hg38 can be specified by argument `genome
= "hg38"`. To use the older versions such as 3.0.0, specify as
submitGreatJob(..., version = "3"). Set argument `help` to `FALSE` to
turn off this messag

  |===================================================================== |  98%


The default enrichment table does not contain informatin of associated
genes for each input region. You can set `download_by = 'tsv'` to
download the complete table, but note only the top 500 regions can be
retreived. See the following link:

https://great-help.atlassian.net/wiki/spaces/GREAT/pages/655401/Export#Export-GlobalExport

Except the additional gene-region association column if taking 'tsv' as
the source of result, all other columns are the same if you choose
'json' (the default) as the source. Or you can try the local GREAT
analysis with the function `great()`.

  ✅ GREAT results saved.


--- Running GREAT for Module 5 ---

Note: On Aug 19 2019 GREAT released version 4 which supports hg38
genome and removes some ontologies such pathways. submitGreatJob()
still takes hg19 as default. hg38 can be specified by argument `genome
= "hg38"`. To use the older versions such as 3.0.0, specify as
submitGreatJob(..., version = "3"). Set argument `help` to `FALSE` to
turn off this messag

  |===================================================================== |  99%


The default enrichment table does not contain informatin of associated
genes for each input region. You can set `download_by = 'tsv'` to
download the complete table, but note only the top 500 regions can be
retreived. See the following link:

https://great-help.atlassian.net/wiki/spaces/GREAT/pages/655401/Export#Export-GlobalExport

Except the additional gene-region association column if taking 'tsv' as
the source of result, all other columns are the same if you choose
'json' (the default) as the source. Or you can try the local GREAT
analysis with the function `great()`.

  ✅ GREAT results saved.


--- Running GSEA for Cell Type: Colonocytes ---

no term enriched under specific pvalueCutoff...


--- Running GSEA for Cell Type: Crypt.Fibroblasts.WNT2B. ---

no term enriched under specific pvalueCutoff...


--- Running GSEA for Cell Type: ECs ---

no term enriched under specific pvalueCutoff...


--- Running GSEA for Cell Type: Enterocytes ---

Warning message in fgseaMultilevel(pathway

In [25]:
# =============================================================================
# Cell 13: Compute Motif Enrichment & SAVE FULL OBJECTS
# =============================================================================
suppressPackageStartupMessages({
  library(monaLisa)
  library(JASPAR2022)
  library(TFBSTools)
  library(BSgenome.Hsapiens.UCSC.hg38)
  library(Biostrings)
})

message("\n=== Step 1: Compute and Save Motif Enrichment ===")

# 1. Setup Directories & Load Motifs
motif_dir <- file.path(OUT_DIR, "differential_results", "motifs")
dir.create(motif_dir, showWarnings = FALSE, recursive = TRUE)

message("Loading JASPAR database...")
pwms <- getMatrixSet(JASPAR2022, opts = list(tax_group = "vertebrates", matrixtype = "PWM"))

# 2. The Core Function (Returns the FULL object)
run_binned_enrichment <- function(peak_list_named, pwms_obj, genome_obj) {
  # Map peaks to groups
  mapping_df <- data.frame(
    peak_id = unlist(peak_list_named, use.names = FALSE),
    group = rep(names(peak_list_named), lengths(peak_list_named))
  )
  
  # Fetch coordinates and sequences
  coords <- get_peak_info(mapping_df$peak_id, "Human", "coordinates")
  mapping_df <- mapping_df[mapping_df$peak_id %in% coords$name, ]
  rownames(coords) <- coords$name
  coords <- coords[mapping_df$peak_id, ] 
  
  gr <- makeGRangesFromDataFrame(coords)
  seqs <- getSeq(genome_obj, gr)
  group_factor <- factor(mapping_df$group, levels = names(peak_list_named))
  
  message("Running binned enrichment for ", length(levels(group_factor)), " groups...")
  
  # Compute Enrichment
  se_enrich <- calcBinnedMotifEnrR(seqs = seqs, 
                                   bins = group_factor,
                                   pwmL = pwms_obj,       
                                   min.score = 10,        
                                   verbose = FALSE)
  return(se_enrich)
}

# -----------------------------------------------------------------------------
# Part A: Cell Types
# -----------------------------------------------------------------------------
message("\n--- Processing Cell Types ---")
ct_peaks <- list()
for (ct in names(res_list)) {
  if ("Human" %in% names(res_list[[ct]])) {
    res_df <- as.data.frame(res_list[[ct]][["Human"]])
    up_peaks <- rownames(res_df)[!is.na(res_df$padj) & res_df$padj < 0.05 & res_df$log2FoldChange > 1]
    if (length(up_peaks) >= 20) ct_peaks[[ct]] <- up_peaks
  }
}

se_ct <- run_binned_enrichment(ct_peaks, pwms, BSgenome.Hsapiens.UCSC.hg38)
saveRDS(se_ct, file.path(motif_dir, "CellType_SE_Object.rds"))
message("💾 Checkpoint: CellType_SE_Object.rds saved!")

# -----------------------------------------------------------------------------
# Part B: Modules
# -----------------------------------------------------------------------------
message("\n--- Processing Modules ---")
mod_peaks_list <- split(master_df$region_id, master_df$module)
mod_peaks_list <- mod_peaks_list[sapply(mod_peaks_list, length) >= 20]

se_mod <- run_binned_enrichment(mod_peaks_list, pwms, BSgenome.Hsapiens.UCSC.hg38)
saveRDS(se_mod, file.path(motif_dir, "Module_SE_Object.rds"))
message("💾 Checkpoint: Module_SE_Object.rds saved!")

message("🎉 Heavy computations finished and safely stored on disk.")


=== Step 1: Compute and Save Motif Enrichment ===

Loading JASPAR database...


--- Processing Cell Types ---

Running binned enrichment for 13 groups...

💾 Checkpoint: CellType_SE_Object.rds saved!


--- Processing Modules ---

Running binned enrichment for 40 groups...

💾 Checkpoint: Module_SE_Object.rds saved!

🎉 Heavy computations finished and safely stored on disk.



In [26]:
# =============================================================================
# Cell 14: Plot ComplexHeatmap from Saved Objects
# =============================================================================
suppressPackageStartupMessages({
  library(ComplexHeatmap)
  library(circlize)
  library(SummarizedExperiment)
})

message("\n=== Step 2: Plotting ComplexHeatmap ===")

# 1. Load the rich objects from disk
motif_dir <- file.path(OUT_DIR, "differential_results", "motifs")
se_ct <- readRDS(file.path(motif_dir, "CellType_SE_Object.rds"))
se_mod <- readRDS(file.path(motif_dir, "Module_SE_Object.rds"))

# 2. Extract and clean the numeric matrices
extract_clean_matrix <- function(se_obj) {
  mat <- assay(se_obj, "negLog10Padj")
  rownames(mat) <- gsub(".*_", "", rownames(mat)) # Clean JASPAR names
  return(mat)
}
ct_enr_mat <- extract_clean_matrix(se_ct)
mod_enr_mat <- extract_clean_matrix(se_mod)

# 3. Plotting Function
plot_motif_heatmap <- function(mat, title, filename) {
  # Keep top 3 per column for a clean summary
  top_indices <- unique(as.vector(apply(mat, 2, function(x) order(x, decreasing = TRUE)[1:3])))
  plot_sub <- mat[top_indices, ]
  plot_sub[plot_sub > 20] <- 20 # Cap significance
  
  col_fun <- colorRamp2(c(0, 2, 20), c("white", "orange", "red"))
  
  ht <- Heatmap(plot_sub, 
                name = "Enrichment", # Safe viewport name
                col = col_fun,
                column_title = title,
                cluster_rows = TRUE, 
                cluster_columns = TRUE,
                show_row_names = TRUE,
                border = TRUE,
                column_names_rot = 45,
                heatmap_legend_param = list(title = "-log10(p-adj)")
  )
  
  pdf(filename, width = 12, height = 14)
  draw(ht, merge_legend = TRUE)
  dev.off()
  message("  ✅ Saved: ", basename(filename))
}

plot_motif_heatmap(ct_enr_mat, "Top Motifs per Human-UP Cell Type", file.path(OUT_DIR, "plots", "Motif_vs_CellType_Heatmap.pdf"))
plot_motif_heatmap(mod_enr_mat, "Top Motifs per K-Means Module", file.path(OUT_DIR, "plots", "Motif_vs_Module_Heatmap.pdf"))


=== Step 2: Plotting ComplexHeatmap ===

  ✅ Saved: Motif_vs_CellType_Heatmap.pdf

  ✅ Saved: Motif_vs_Module_Heatmap.pdf



In [27]:
# =============================================================================
# Cell 15: Plot Native monaLisa with Sequence Logos
# =============================================================================
suppressPackageStartupMessages({
  library(monaLisa)
  library(SummarizedExperiment)
})

message("\n=== Step 3: Plotting Native monaLisa with Logos ===")

# 1. Load the rich objects from disk
motif_dir <- file.path(OUT_DIR, "differential_results", "motifs")
se_ct <- readRDS(file.path(motif_dir, "CellType_SE_Object.rds"))
se_mod <- readRDS(file.path(motif_dir, "Module_SE_Object.rds"))

# 2. Plotting Function
plot_monalisa_native <- function(se_obj, filename, top_n = 30) {
  # Filter for top overall motifs to prevent PDF rendering crashes
  enr_mat <- assay(se_obj, "negLog10Padj")
  max_enr <- apply(enr_mat, 1, max, na.rm = TRUE)
  top_motif_idx <- order(max_enr, decreasing = TRUE)[1:top_n]
  se_sub <- se_obj[top_motif_idx, ]
  
  # Draw the plot
  pdf(filename, width = 16, height = 12) 
  plotMotifHeatmaps(x = se_sub, 
                    which.plots = c("enr", "FDR"), 
                    cluster = TRUE,
                    show_motif_GC = TRUE,
                    show_seqlogo = TRUE)
  dev.off()
  message("  ✅ Saved: ", basename(filename))
}

plot_monalisa_native(se_ct, file.path(OUT_DIR, "plots", "Native_monaLisa_CellTypes.pdf"), top_n = 35)
plot_monalisa_native(se_mod, file.path(OUT_DIR, "plots", "Native_monaLisa_Modules.pdf"), top_n = 35)


=== Step 3: Plotting Native monaLisa with Logos ===

Warning message in FUN(newX[, i], ...):
“no non-missing arguments to max; returning -Inf”


ERROR: Error in plotMotifHeatmaps(x = se_sub, which.plots = c("enr", "FDR"), : all(which.plots %in% assayNames(x)) is not TRUE


In [34]:
# =============================================================================
# Cell 15: Plot Native monaLisa with Sequence Logos (Labels Safely Fixed)
# =============================================================================
suppressPackageStartupMessages({
  library(monaLisa)
  library(SummarizedExperiment)
})

message("\n=== Step 3: Plotting Native monaLisa with Logos ===")

motif_dir <- file.path(OUT_DIR, "differential_results", "motifs")

# Load your safely saved objects
se_ct <- readRDS(file.path(motif_dir, "CellType_SE_Object.rds"))
se_mod <- readRDS(file.path(motif_dir, "Module_SE_Object.rds"))

# Corrected Plotting Function
plot_monalisa_native <- function(se_obj, filename, top_n = 30) {
  
  # 1. Safely calculate the max enrichment
  enr_mat <- assay(se_obj, "negLog10Padj")
  enr_mat[is.na(enr_mat)] <- 0 # Convert NAs to 0
  
  max_enr <- apply(enr_mat, 1, max)
  top_motif_idx <- order(max_enr, decreasing = TRUE)[1:top_n]
  se_sub <- se_obj[top_motif_idx, ]
  
  # 2. Draw the plot safely
  pdf(filename, width = 16, height = 12) 
  plotMotifHeatmaps(x = se_sub, 
                    which.plots = c("log2enr", "negLog10Padj"), 
                    cluster = TRUE,
                    show_motif_GC = FALSE, 
                    show_seqlogo = TRUE,
                    
                    # 👈 THE FIX: We remove `show_column_names = TRUE` to avoid the crash, 
                    # but keep the rotation. ComplexHeatmap will now auto-draw them at 
                    # 45 degrees so they don't get cut off the bottom of the page!
                    column_names_rot = 45,
                    column_names_gp = grid::gpar(fontsize = 12))
  dev.off()
  
  message("  ✅ Saved: ", basename(filename))
}

plot_monalisa_native(se_ct, file.path(OUT_DIR, "plots", "Native_monaLisa_CellTypes.pdf"), top_n = 35)
plot_monalisa_native(se_mod, file.path(OUT_DIR, "plots", "Native_monaLisa_Modules.pdf"), top_n = 35)

message("🎉 Sequence logo heatmaps successfully rendered with visible labels!")


=== Step 3: Plotting Native monaLisa with Logos ===

Warning message in grabDL(warn, wrap, wrap.grobs, ...):
“one or more grobs overwritten (grab WILL not be faithful; try 'wrap.grobs = TRUE')”
  ✅ Saved: Native_monaLisa_CellTypes.pdf

Warning message in grabDL(warn, wrap, wrap.grobs, ...):
“one or more grobs overwritten (grab WILL not be faithful; try 'wrap.grobs = TRUE')”
  ✅ Saved: Native_monaLisa_Modules.pdf

🎉 Sequence logo heatmaps successfully rendered with visible labels!



In [39]:
# =============================================================================
# Cell 15: Plot Native monaLisa with Sequence Logos (The True Fix)
# =============================================================================
suppressPackageStartupMessages({
  library(monaLisa)
  library(SummarizedExperiment)
  library(ComplexHeatmap)
})

message("\n=== Step 3: Plotting Native monaLisa with Logos ===")

motif_dir <- file.path(OUT_DIR, "differential_results", "motifs")

# Load your safely saved objects
se_ct <- readRDS(file.path(motif_dir, "CellType_SE_Object.rds"))
se_mod <- readRDS(file.path(motif_dir, "Module_SE_Object.rds"))

# Corrected Plotting Function
plot_monalisa_native <- function(se_obj, filename, top_n = 30) {
  
  # 1. Safely calculate the max enrichment
  enr_mat <- assay(se_obj, "negLog10Padj")
  enr_mat[is.na(enr_mat)] <- 0 # Convert NAs to 0
  
  max_enr <- apply(enr_mat, 1, max)
  top_motif_idx <- order(max_enr, decreasing = TRUE)[1:top_n]
  se_sub <- se_obj[top_motif_idx, ]
  
  # 2. Generate the raw list of heatmaps (DO NOT alter colnames)
  hm_list <- plotMotifHeatmaps(x = se_sub, 
                               which.plots = c("log2enr", "negLog10Padj"), 
                               cluster = TRUE,
                               show_motif_GC = FALSE, 
                               show_seqlogo = TRUE,
                               column_names_rot = 45,
                               column_names_gp = grid::gpar(fontsize = 10))
  
  # 3. THE FIX: Replicate monaLisa's internal merging step 
  # This converts the raw list into a proper ComplexHeatmap 'HeatmapList' object
  ht_list <- Reduce(ComplexHeatmap::add_heatmap, hm_list)
  
  # 4. Draw the merged object with our massive bottom padding!
  pdf(filename, width = 16, height = 12) 
  draw(ht_list, padding = unit(c(2, 2, 40, 2), "mm"))
  dev.off()
  
  message("  ✅ Saved: ", basename(filename))
}

plot_monalisa_native(se_ct, file.path(OUT_DIR, "plots", "Native_monaLisa_CellTypes.pdf"), top_n = 35)
plot_monalisa_native(se_mod, file.path(OUT_DIR, "plots", "Native_monaLisa_Modules.pdf"), top_n = 35)

message("🎉 Sequence logo heatmaps successfully rendered!")


=== Step 3: Plotting Native monaLisa with Logos ===

Warning message in grabDL(warn, wrap, wrap.grobs, ...):
“one or more grobs overwritten (grab WILL not be faithful; try 'wrap.grobs = TRUE')”
Warning message in grabDL(warn, wrap, wrap.grobs, ...):
“one or more grobs overwritten (grab WILL not be faithful; try 'wrap.grobs = TRUE')”
  ✅ Saved: Native_monaLisa_CellTypes.pdf

Warning message in grabDL(warn, wrap, wrap.grobs, ...):
“one or more grobs overwritten (grab WILL not be faithful; try 'wrap.grobs = TRUE')”
Warning message in grabDL(warn, wrap, wrap.grobs, ...):
“one or more grobs overwritten (grab WILL not be faithful; try 'wrap.grobs = TRUE')”
  ✅ Saved: Native_monaLisa_Modules.pdf

🎉 Sequence logo heatmaps successfully rendered!



In [42]:
# =============================================================================
# Quick Recovery: Rebuild and Save Pseudobulk Data
# =============================================================================
library(dplyr)
library(arrow)

message("\n=== Rebuilding pb_counts and pb_meta from Parquet ===")

# 1. Load Parquet Files
all_counts <- list()
all_meta <- list()

for (sp in SPECIES) {
  sp_dir <- file.path(QUANT_DIR, sp)
  counts_file <- file.path(sp_dir, "pseudobulk_counts.parquet")
  meta_file <- file.path(sp_dir, "pseudobulk_groups.parquet") 
  
  if (!file.exists(counts_file)) next
  
  counts <- as.data.frame(read_parquet(counts_file))
  rownames(counts) <- counts$region_id
  counts$region_id <- NULL
  
  meta <- as.data.frame(read_parquet(meta_file))
  new_labels <- paste0(sp, "_", meta$label)
  colnames(counts) <- new_labels
  meta$sample_id <- new_labels
  meta$species <- sp
  
  all_counts[[sp]] <- counts
  all_meta[[sp]] <- meta
}

# 2. Merge
shared_peaks <- Reduce(intersect, lapply(all_counts, rownames))
counts_merged <- do.call(cbind, unname(lapply(all_counts, function(x) x[shared_peaks, , drop = FALSE])))
counts_merged[is.na(counts_merged)] <- 0

meta_merged <- do.call(rbind, unname(all_meta))
rownames(meta_merged) <- meta_merged$sample_id
counts_merged <- counts_merged[, rownames(meta_merged)]

# 3. Filter for Adults & Quality
meta_merged <- meta_merged[meta_merged$age == "Adult", ]
counts_merged <- counts_merged[, rownames(meta_merged)]

meta_merged$total_counts <- colSums(counts_merged)
keep_samples <- meta_merged$total_counts >= 50000 & meta_merged$n_cells >= 100

meta_filtered <- meta_merged[keep_samples, ]
counts_filtered <- counts_merged[, rownames(meta_filtered)]

# 4. Filter for Shared Cell Types
meta_filtered$species <- factor(meta_filtered$species, levels = SPECIES)
meta_filtered$cell_type <- as.factor(make.names(as.character(meta_filtered$cell_type)))

ct_per_species <- split(as.character(meta_filtered$cell_type), meta_filtered$species)
shared_ct <- Reduce(intersect, ct_per_species)

keep_shared <- meta_filtered$cell_type %in% shared_ct
counts_shared <- counts_filtered[, keep_shared]
meta_shared <- meta_filtered[keep_shared, ]

# 5. Species-Aware Peak Filtering (The final step from Cell 6)
keep_count <- apply(counts_shared, 1, max) >= 150
active_in_species <- sapply(SPECIES, function(sp) {
  sp_cols <- meta_shared$species == sp
  if (sum(sp_cols) > 0) rowSums(counts_shared[, sp_cols, drop = FALSE] >= 10) >= 2
  else rep(FALSE, nrow(counts_shared))
})
keep_peaks <- keep_count & (rowSums(active_in_species) >= 6)

# 6. Final Assignment & Saving
# Convert to a list of matrices split by cell type (which is what Cell 16 expects)
pb_counts <- list()
pb_meta <- meta_shared

for (ct in unique(meta_shared$cell_type)) {
  ct_samples <- rownames(meta_shared)[meta_shared$cell_type == ct]
  pb_counts[[ct]] <- counts_shared[keep_peaks, ct_samples, drop = FALSE]
}

# Save them!
save_dir <- file.path(OUT_DIR, "pseudobulk")
dir.create(save_dir, showWarnings = FALSE)
saveRDS(pb_counts, file.path(save_dir, "pb_counts_by_celltype.rds"))
saveRDS(pb_meta, file.path(save_dir, "pb_meta_filtered.rds"))

message("✅ Recovery Complete: pb_counts and pb_meta rebuilt and saved!")


=== Rebuilding pb_counts and pb_meta from Parquet ===

✅ Recovery Complete: pb_counts and pb_meta rebuilt and saved!



In [51]:
# =============================================================================
# Cell 18: Ultra-Robust Human-Specific Filtering (Perfect Separation)
# =============================================================================
library(matrixStats)

message("\n=== Running Ultra-Robust Sample-Level Filtering ===")

robust_dir <- file.path(OUT_DIR, "differential_results", "ultra_robust_peaks")
dir.create(robust_dir, showWarnings = FALSE, recursive = TRUE)

robust_summary <- list()

for (ct in names(pb_counts)) {
  if (!"Human" %in% names(res_list[[ct]])) next
  
  message("\nProcessing: ", ct)
  
  # Extract significant DESeq2 Human-UP peaks
  res_df <- as.data.frame(res_list[[ct]][["Human"]])
  deseq_up <- rownames(res_df)[!is.na(res_df$padj) & res_df$padj < 0.05 & res_df$log2FoldChange > 1]
  
  # 👈 THE FIX: Intersect with available peaks to prevent 'NA' ghost rows!
  deseq_up <- intersect(deseq_up, rownames(pb_counts[[ct]]))
  
  if (length(deseq_up) == 0) {
    message("  ⏭️ No DESeq2 UP peaks to filter.")
    next
  }
  
  # Get raw counts and metadata
  counts <- pb_counts[[ct]][deseq_up, , drop = FALSE]
  meta <- pb_meta[pb_meta$cell_type == ct, ]
  
  common_samples <- intersect(rownames(meta), colnames(counts))
  counts <- counts[, common_samples, drop = FALSE]
  meta <- meta[common_samples, ]
  
  # Normalize to Log2(CPM + 1) using the full library sizes
  lib_sizes <- colSums(pb_counts[[ct]][, common_samples, drop = FALSE]) 
  cpm <- t(t(counts) / lib_sizes) * 1e6
  log_cpm <- log2(cpm + 1)
  
  # 👈 LOWERCASE FIX: 'species' instead of 'Species'
  human_samples <- rownames(meta)[meta$species == "Human"]
  nonhuman_samples <- rownames(meta)[meta$species != "Human"]
  
  if (length(human_samples) < 2 || length(nonhuman_samples) < 1) {
    message("  ⚠️ Not enough donors to test robustness. Skipping.")
    next
  }
  
  human_mat <- log_cpm[, human_samples, drop = FALSE]
  nonhuman_mat <- log_cpm[, nonhuman_samples, drop = FALSE]
  
  min_human <- rowMins(as.matrix(human_mat))
  max_nonhuman <- rowMaxs(as.matrix(nonhuman_mat))
  
  # The Brutal Filter
  is_ultra_robust <- (min_human > max_nonhuman) & (min_human > 1)
  is_ultra_robust[is.na(is_ultra_robust)] <- FALSE # Catch any stray NAs
  
  ultra_robust_peaks <- rownames(log_cpm)[is_ultra_robust]
  
  message(sprintf("  ✅ DESeq2 Peaks: %d | Ultra-Robust Surviving: %d", 
                  length(deseq_up), length(ultra_robust_peaks)))
  
  # Save the results safely
  if (length(ultra_robust_peaks) > 0) {
    out_df <- data.frame(
      peak_id = ultra_robust_peaks,
      min_Human_logCPM = min_human[is_ultra_robust],
      max_NonHuman_logCPM = max_nonhuman[is_ultra_robust]
    )
    
    out_df <- cbind(out_df, log_cpm[ultra_robust_peaks, , drop = FALSE])
    
    write.csv(out_df, file.path(robust_dir, paste0(ct, "_Ultra_Robust_Human_UP.csv")), row.names = FALSE)
    
    robust_bed <- get_peak_info(ultra_robust_peaks, "Human", "coordinates")
    write.table(robust_bed, file.path(robust_dir, paste0(ct, "_Ultra_Robust.bed")), 
                sep = "\t", quote = FALSE, row.names = FALSE, col.names = FALSE)
    
    robust_summary[[ct]] <- length(ultra_robust_peaks)
  }
}

message("\n🎉 Ultra-Robust filtering complete! Files saved to: ", robust_dir)


=== Running Ultra-Robust Sample-Level Filtering ===


Processing: Colonocytes

  ✅ DESeq2 Peaks: 4627 | Ultra-Robust Surviving: 926


Processing: Crypt.Fibroblasts.WNT2B.

  ✅ DESeq2 Peaks: 2407 | Ultra-Robust Surviving: 108


Processing: ECs

  ✅ DESeq2 Peaks: 165 | Ultra-Robust Surviving: 42


Processing: Enterocytes

  ✅ DESeq2 Peaks: 5274 | Ultra-Robust Surviving: 1599


Processing: Goblet.cells

  ✅ DESeq2 Peaks: 4976 | Ultra-Robust Surviving: 60


Processing: Macrophages

  ✅ DESeq2 Peaks: 1649 | Ultra-Robust Surviving: 67


Processing: Naive.B.cells

  ✅ DESeq2 Peaks: 759 | Ultra-Robust Surviving: 133


Processing: Plasma.B.cells

  ✅ DESeq2 Peaks: 2911 | Ultra-Robust Surviving: 88


Processing: Specialized.Fibroblasts.RSPO3..only

  ✅ DESeq2 Peaks: 890 | Ultra-Robust Surviving: 117


Processing: Specialized.Fibroblasts.SYNM.

  ✅ DESeq2 Peaks: 2966 | Ultra-Robust Surviving: 195


Processing: Stem.cells

  ✅ DESeq2 Peaks: 3254 | Ultra-Robust Surviving: 122


Processing: T.cells

In [54]:
# =============================================================================
# Cell 16: Evolutionary Branch Differential Testing (Phylogenetic Contrasts)
# =============================================================================
library(DESeq2)
library(dplyr)
library(tibble)

message("\n=== Running Evolutionary Branch Testing ===")

# 1. Define the Evolutionary Splits (Nodes of the Tree)
# =============================================================================
# The Strict "Synapomorphy" Branch Definitions
# Replace the 'branches' list at the top of Cell 16 with this:
# =============================================================================

branches <- list(
  # Node 1: Human vs ALL other primates
  "Node1_Human_Gain" = list(
    pos = c("Human"), 
    neg = c("Chimpanzee", "Bonobo", "Gorilla", "Macaque", "Marmoset")
  ),
  
  # Node 2: African Apes vs ALL older primates
  "Node2_AfricanApe_Gain" = list(
    pos = c("Human", "Chimpanzee", "Bonobo"), 
    neg = c("Gorilla", "Macaque", "Marmoset")
  ),
  
  # Node 3: Great Apes vs ALL older primates
  "Node3_GreatApe_Gain" = list(
    pos = c("Human", "Chimpanzee", "Bonobo", "Gorilla"), 
    neg = c("Macaque", "Marmoset")
  ),
  
  # Node 4: Old World vs Marmoset (unchanged, as Marmoset is the root outgroup)
  "Node4_OldWorld_Gain" = list(
    pos = c("Human", "Chimpanzee", "Bonobo", "Gorilla", "Macaque"), 
    neg = c("Marmoset")
  )
)

branch_dir <- file.path(OUT_DIR, "differential_results", "evolutionary_branches")
dir.create(branch_dir, showWarnings = FALSE, recursive = TRUE)

# 2. Helper function to create safe numeric contrast vectors
make_contrast_vector <- function(sp_pos, sp_neg, res_names) {
  vec <- rep(0, length(res_names))
  names(vec) <- res_names
  
  # 👈 THE FIX: lowercase "species" to match DESeq2 coefficient names
  pos_names <- paste0("species", sp_pos) 
  neg_names <- paste0("species", sp_neg)
  
  vec[pos_names] <- 1 / length(pos_names)
  vec[neg_names] <- -1 / length(neg_names)
  
  return(vec)
}

branch_res_list <- list()

# 3. Loop over each Cell Type
for (ct in names(pb_counts)) {
  message("\nProcessing cell type: ", ct)
  
  counts <- pb_counts[[ct]]
  meta <- pb_meta[pb_meta$cell_type == ct, ]
  
  common_samples <- intersect(rownames(meta), colnames(counts))
  counts <- counts[, common_samples, drop = FALSE]
  meta <- meta[common_samples, ]
  
  # 👈 THE FIX: lowercase meta$species
  if (length(unique(meta$species)) < 2 || ncol(counts) < 4) {
    message("  ⚠️ Skipping (Not enough species/samples)")
    next
  }
  
  keep <- rowSums(counts >= 5) >= 2
  counts <- counts[keep, ]
  
  # Set up the Zero-Intercept Model (using lowercase species)
  meta$species <- factor(meta$species)
  dds <- DESeqDataSetFromMatrix(countData = counts, colData = meta, design = ~ 0 + species)
  
  # 👈 THE FIX: Use poscounts to handle dropouts in ATAC-seq data!
  dds <- estimateSizeFactors(dds, type = "poscounts")
  dds <- suppressMessages(DESeq(dds))
  res_names <- resultsNames(dds)
  
  branch_res_list[[ct]] <- list()
  
  # 4. Run the contrasts for each evolutionary node
  for (branch_name in names(branches)) {
    pos_sp <- branches[[branch_name]]$pos
    neg_sp <- branches[[branch_name]]$neg
    
    # Check if the required species for this contrast exist in this cell type
    available_sp <- levels(meta$species)
    if (all(c(pos_sp, neg_sp) %in% available_sp)) {
      
      contrast_vec <- make_contrast_vector(pos_sp, neg_sp, res_names)
      res <- results(dds, contrast = contrast_vec, alpha = 0.05)
      res_df <- as.data.frame(res)
      
      branch_res_list[[ct]][[branch_name]] <- res_df
      up_peaks <- rownames(res_df)[!is.na(res_df$padj) & res_df$padj < 0.05 & res_df$log2FoldChange > 1]
      
      message(sprintf("  ✅ %s: Found %d defining peaks", branch_name, length(up_peaks)))
      
      if (length(up_peaks) > 0) {
        ct_branch_dir <- file.path(branch_dir, ct)
        dir.create(ct_branch_dir, showWarnings = FALSE)
        write.csv(res_df[up_peaks, ], file.path(ct_branch_dir, paste0(branch_name, "_UP_Stats.csv")))
      }
      
    } else {
      message("  ⏭️ Skipping ", branch_name, " (Missing required species in this cell type)")
    }
  }
}

saveRDS(branch_res_list, file.path(branch_dir, "Evolutionary_Branch_DESeq2_res.rds"))
message("\n🎉 Evolutionary Branch Testing Complete!")


=== Running Evolutionary Branch Testing ===


Processing cell type: Colonocytes

  ✅ Node1_Human_Gain: Found 5307 defining peaks

  ✅ Node2_AfricanApe_Gain: Found 2449 defining peaks

  ✅ Node3_GreatApe_Gain: Found 4724 defining peaks

  ✅ Node4_OldWorld_Gain: Found 4104 defining peaks


Processing cell type: Crypt.Fibroblasts.WNT2B.

  ✅ Node1_Human_Gain: Found 3682 defining peaks

  ✅ Node2_AfricanApe_Gain: Found 1206 defining peaks

  ✅ Node3_GreatApe_Gain: Found 2196 defining peaks

  ✅ Node4_OldWorld_Gain: Found 2772 defining peaks


Processing cell type: ECs

  ✅ Node1_Human_Gain: Found 241 defining peaks

  ✅ Node2_AfricanApe_Gain: Found 27 defining peaks

  ✅ Node3_GreatApe_Gain: Found 29 defining peaks

  ✅ Node4_OldWorld_Gain: Found 479 defining peaks


Processing cell type: Enterocytes

  ✅ Node1_Human_Gain: Found 5506 defining peaks

  ✅ Node2_AfricanApe_Gain: Found 2640 defining peaks

  ✅ Node3_GreatApe_Gain: Found 4636 defining peaks

  ✅ Node4_OldWorld_Gain: Found 6182

In [55]:
# =============================================================================
# Cell 17: Visualizing Evolutionary Branch Effects (Staircase Heatmap)
# =============================================================================
suppressPackageStartupMessages({
  library(ComplexHeatmap)
  library(circlize)
})

message("\n=== Generating Evolutionary Branch Heatmap ===")

TARGET_CT <- "Enterocytes" 

if (!TARGET_CT %in% names(branch_res_list)) {
  stop("Target cell type not found in branch results!")
}

counts <- pb_counts[[TARGET_CT]]
meta <- pb_meta[pb_meta$cell_type == TARGET_CT, ]

common_samples <- intersect(rownames(meta), colnames(counts))
counts <- counts[, common_samples, drop = FALSE]
meta <- meta[common_samples, ]

lib_sizes <- colSums(counts)
cpm <- t(t(counts) / lib_sizes) * 1e6
log_cpm <- log2(cpm + 1)

# 3. Calculate Average Expression per Species (Bulletproof Matrix method)
species_order <- c("Human", "Chimpanzee", "Bonobo", "Gorilla", "Macaque", "Marmoset")

avg_exp <- matrix(NA, nrow = nrow(log_cpm), ncol = length(species_order))
rownames(avg_exp) <- rownames(log_cpm) # 👈 Explicitly preserve the Peak IDs
colnames(avg_exp) <- species_order

for (sp in species_order) {
  sp_samples <- rownames(meta)[meta$species == sp] # 👈 Lowercase 's'
  if (length(sp_samples) > 1) {
    avg_exp[, sp] <- rowMeans(log_cpm[, sp_samples, drop = FALSE])
  } else if (length(sp_samples) == 1) {
    avg_exp[, sp] <- log_cpm[, sp_samples]
  }
}

# 4. Extract the Top 50 Peaks per Branch
plot_peaks <- c()
row_splits <- c()

for (branch in names(branches)) {
  if (branch %in% names(branch_res_list[[TARGET_CT]])) {
    res <- branch_res_list[[TARGET_CT]][[branch]]
    sig_peaks <- rownames(res)[!is.na(res$padj) & res$padj < 0.05 & res$log2FoldChange > 1]
    sig_res <- res[sig_peaks, , drop = FALSE]
    
    top_peaks <- rownames(sig_res)[order(sig_res$log2FoldChange, decreasing = TRUE)]
    top_peaks <- head(top_peaks, 50)
    
    if (length(top_peaks) > 0) {
      plot_peaks <- c(plot_peaks, top_peaks)
      row_splits <- c(row_splits, rep(branch, length(top_peaks)))
    }
  }
}

message(sprintf("Found %d total defining peaks across branches for %s.", length(plot_peaks), TARGET_CT))

# 5. Build the Z-score Matrix
mat <- avg_exp[plot_peaks, , drop = FALSE]
valid_cols <- apply(mat, 2, function(x) !all(is.na(x)))
mat <- mat[, valid_cols, drop = FALSE]

mat_scaled <- t(apply(mat, 1, scale))
colnames(mat_scaled) <- colnames(mat)
split_factor <- factor(row_splits, levels = names(branches))

# 6. Draw the ComplexHeatmap
col_fun <- colorRamp2(c(-2, 0, 2), c("#377eb8", "white", "#e41a1c"))

ht <- Heatmap(mat_scaled, name = "Z-score", col = col_fun,
              row_split = split_factor, row_title_rot = 0, row_title_gp = gpar(fontsize = 10, fontface = "bold"),
              row_gap = unit(2, "mm"), cluster_rows = TRUE, cluster_columns = FALSE, 
              show_row_names = FALSE, column_names_rot = 45, 
              column_title = paste("Evolutionary Branch Gains -", TARGET_CT),
              column_title_gp = gpar(fontsize = 14, fontface = "bold"), border = TRUE)

plot_dir <- file.path(OUT_DIR, "plots", "evolutionary_branches")
dir.create(plot_dir, showWarnings = FALSE, recursive = TRUE)
plot_file <- file.path(plot_dir, paste0(TARGET_CT, "_Evolutionary_Staircase_Heatmap.pdf"))

pdf(plot_file, width = 8, height = 10)
draw(ht, merge_legend = TRUE)
dev.off()
message("✅ Evolutionary Staircase Heatmap saved to: ", plot_file)


=== Generating Evolutionary Branch Heatmap ===

Found 200 total defining peaks across branches for Enterocytes.



pdf 
  2

✅ Evolutionary Staircase Heatmap saved to: /cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/13_deseq2_R_pseudobulk/plots/evolutionary_branches/Enterocytes_Evolutionary_Staircase_Heatmap.pdf



In [56]:
# =============================================================================
# Cell 19: Ultra-Robust Filtering for Evolutionary Branches
# =============================================================================
library(matrixStats)

message("\n=== Running Ultra-Robust Filtering on Branches ===")

robust_branch_dir <- file.path(OUT_DIR, "differential_results", "ultra_robust_branches")
dir.create(robust_branch_dir, showWarnings = FALSE, recursive = TRUE)

robust_branch_peaks <- list() # Store for motif enrichment

for (ct in names(branch_res_list)) {
  message("\nProcessing: ", ct)
  robust_branch_peaks[[ct]] <- list()
  
  # Get raw counts and metadata
  counts <- pb_counts[[ct]]
  meta <- pb_meta[pb_meta$cell_type == ct, ]
  common_samples <- intersect(rownames(meta), colnames(counts))
  counts <- counts[, common_samples, drop = FALSE]
  meta <- meta[common_samples, ]
  
  # Log2CPM normalization
  lib_sizes <- colSums(pb_counts[[ct]][, common_samples])
  cpm <- t(t(counts) / lib_sizes) * 1e6
  log_cpm <- log2(cpm + 1)
  
  for (branch in names(branches)) {
    # 1. Get the DESeq2 significant peaks for this branch
    if (!branch %in% names(branch_res_list[[ct]])) next
    
    res_df <- branch_res_list[[ct]][[branch]]
    deseq_up <- rownames(res_df)[!is.na(res_df$padj) & res_df$padj < 0.05 & res_df$log2FoldChange > 1]
    
    if (length(deseq_up) == 0) next
    
    # 2. Identify Positive and Negative Samples
    pos_sp <- branches[[branch]]$pos
    neg_sp <- branches[[branch]]$neg
    
    pos_samples <- rownames(meta)[meta$species %in% pos_sp]
    neg_samples <- rownames(meta)[meta$species %in% neg_sp]
    
    if (length(pos_samples) < 1 || length(neg_samples) < 1) next
    
    # 3. Apply the Ultra-Robust Filter: min(POS donors) > max(NEG donors)
    pos_mat <- log_cpm[deseq_up, pos_samples, drop = FALSE]
    neg_mat <- log_cpm[deseq_up, neg_samples, drop = FALSE]
    
    min_pos <- rowMins(as.matrix(pos_mat))
    max_neg <- rowMaxs(as.matrix(neg_mat))
    
    is_ultra_robust <- (min_pos > max_neg) & (min_pos > 1)
    surviving_peaks <- deseq_up[is_ultra_robust]
    
    message(sprintf("  [%s] %s: DESeq2 = %d | Ultra-Robust = %d", 
                    ct, branch, length(deseq_up), length(surviving_peaks)))
    
    # 4. Save results
    if (length(surviving_peaks) > 0) {
      robust_branch_peaks[[ct]][[branch]] <- surviving_peaks
      
      # Save BED
      bed_df <- get_peak_info(surviving_peaks, "Human", "coordinates")
      ct_dir <- file.path(robust_branch_dir, ct)
      dir.create(ct_dir, showWarnings = FALSE)
      
      write.table(bed_df, file.path(ct_dir, paste0(branch, "_UltraRobust.bed")), 
                  sep = "\t", quote = FALSE, row.names = FALSE, col.names = FALSE)
    }
  }
}
message("✅ Ultra-robust branch filtering complete!")


=== Running Ultra-Robust Filtering on Branches ===


Processing: Colonocytes

  [Colonocytes] Node1_Human_Gain: DESeq2 = 5307 | Ultra-Robust = 1024

  [Colonocytes] Node2_AfricanApe_Gain: DESeq2 = 2449 | Ultra-Robust = 610

  [Colonocytes] Node3_GreatApe_Gain: DESeq2 = 4724 | Ultra-Robust = 2216

  [Colonocytes] Node4_OldWorld_Gain: DESeq2 = 4104 | Ultra-Robust = 2962


Processing: Crypt.Fibroblasts.WNT2B.

  [Crypt.Fibroblasts.WNT2B.] Node1_Human_Gain: DESeq2 = 3682 | Ultra-Robust = 122

  [Crypt.Fibroblasts.WNT2B.] Node2_AfricanApe_Gain: DESeq2 = 1206 | Ultra-Robust = 8

  [Crypt.Fibroblasts.WNT2B.] Node3_GreatApe_Gain: DESeq2 = 2196 | Ultra-Robust = 108

  [Crypt.Fibroblasts.WNT2B.] Node4_OldWorld_Gain: DESeq2 = 2772 | Ultra-Robust = 350


Processing: ECs

  [ECs] Node1_Human_Gain: DESeq2 = 241 | Ultra-Robust = 52

  [ECs] Node2_AfricanApe_Gain: DESeq2 = 27 | Ultra-Robust = 9

  [ECs] Node3_GreatApe_Gain: DESeq2 = 29 | Ultra-Robust = 13

  [ECs] Node4_OldWorld_Gain: DESeq2 = 479 | 

In [57]:
# =============================================================================
# Cell 19.5: Visualizing Ultra-Robust Branches (Staircase Heatmap)
# =============================================================================
suppressPackageStartupMessages({
  library(ComplexHeatmap)
  library(circlize)
})

message("\n=== Generating Ultra-Robust Evolutionary Staircase Heatmap ===")

TARGET_CT <- "Enterocytes" # 👈 Change this to whatever cell type you want to plot

if (!TARGET_CT %in% names(robust_branch_peaks)) {
  stop("Target cell type not found in robust branch results!")
}

# 1. Get raw counts and metadata
counts <- pb_counts[[TARGET_CT]]
meta <- pb_meta[pb_meta$cell_type == TARGET_CT, ]

common_samples <- intersect(rownames(meta), colnames(counts))
counts <- counts[, common_samples, drop = FALSE]
meta <- meta[common_samples, ]

# 2. Normalize
lib_sizes <- colSums(counts)
cpm <- t(t(counts) / lib_sizes) * 1e6
log_cpm <- log2(cpm + 1)

# 3. Calculate Average Expression per Species
species_order <- c("Human", "Chimpanzee", "Bonobo", "Gorilla", "Macaque", "Marmoset")
avg_exp <- matrix(NA, nrow = nrow(log_cpm), ncol = length(species_order))
rownames(avg_exp) <- rownames(log_cpm)
colnames(avg_exp) <- species_order

for (sp in species_order) {
  sp_samples <- rownames(meta)[meta$species == sp]
  if (length(sp_samples) > 1) {
    avg_exp[, sp] <- rowMeans(log_cpm[, sp_samples, drop = FALSE])
  } else if (length(sp_samples) == 1) {
    avg_exp[, sp] <- log_cpm[, sp_samples]
  }
}

# 4. Extract Peaks from the Ultra-Robust List
plot_peaks <- c()
row_splits <- c()

for (branch in names(branches)) {
  if (branch %in% names(robust_branch_peaks[[TARGET_CT]])) {
    ur_peaks <- robust_branch_peaks[[TARGET_CT]][[branch]]
    
    if (length(ur_peaks) > 0) {
      # Grab the original DESeq2 stats so we can sort by Fold Change
      res_df <- branch_res_list[[TARGET_CT]][[branch]]
      ur_res <- res_df[ur_peaks, ]
      
      # Sort by Fold Change and take the top 50
      sorted_ur_peaks <- rownames(ur_res)[order(ur_res$log2FoldChange, decreasing = TRUE)]
      top_peaks <- head(sorted_ur_peaks, 50)
      
      plot_peaks <- c(plot_peaks, top_peaks)
      row_splits <- c(row_splits, rep(branch, length(top_peaks)))
    }
  }
}

if (length(plot_peaks) == 0) {
  stop("No ultra-robust peaks found for this cell type to plot!")
}

message(sprintf("Found %d total ultra-robust defining peaks across branches for %s.", length(plot_peaks), TARGET_CT))

# 5. Build the Z-score Matrix
mat <- avg_exp[plot_peaks, , drop = FALSE]
valid_cols <- apply(mat, 2, function(x) !all(is.na(x)))
mat <- mat[, valid_cols, drop = FALSE]

mat_scaled <- t(apply(mat, 1, scale))
colnames(mat_scaled) <- colnames(mat)
split_factor <- factor(row_splits, levels = names(branches))

# 6. Draw the ComplexHeatmap
col_fun <- colorRamp2(c(-2, 0, 2), c("#377eb8", "white", "#e41a1c"))

ht <- Heatmap(mat_scaled, 
              name = "Z-score", 
              col = col_fun,
              row_split = split_factor, 
              row_title_rot = 0, 
              row_title_gp = gpar(fontsize = 10, fontface = "bold"),
              row_gap = unit(2, "mm"), 
              cluster_rows = TRUE, 
              cluster_columns = FALSE, # Keep species in evolutionary order!
              show_row_names = FALSE, 
              column_names_rot = 45, 
              column_title = paste("Ultra-Robust Branch Gains -", TARGET_CT),
              column_title_gp = gpar(fontsize = 14, fontface = "bold"), 
              border = TRUE)

plot_dir <- file.path(OUT_DIR, "plots", "evolutionary_branches")
dir.create(plot_dir, showWarnings = FALSE, recursive = TRUE)
plot_file <- file.path(plot_dir, paste0(TARGET_CT, "_UltraRobust_Staircase_Heatmap.pdf"))

pdf(plot_file, width = 8, height = 10)
draw(ht, merge_legend = TRUE)
dev.off()

message("✅ Ultra-Robust Staircase Heatmap saved to: ", plot_file)


=== Generating Ultra-Robust Evolutionary Staircase Heatmap ===

Found 200 total ultra-robust defining peaks across branches for Enterocytes.



pdf 
  2

✅ Ultra-Robust Staircase Heatmap saved to: /cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/13_deseq2_R_pseudobulk/plots/evolutionary_branches/Enterocytes_UltraRobust_Staircase_Heatmap.pdf



In [65]:
# =============================================================================
# Cell 21: Dynamic Evolutionary Orthology Index (Full Primate Scope)
# =============================================================================
message("\n=== Building Dynamic Orthology Index ===")

# 1. Map the exact orthology columns from your master annotation
ortho_cols <- list(
  Human      = "Human_orth", 
  Chimpanzee = "Chimpanzee_orth", 
  Bonobo     = "Bonobo_orth",
  Gorilla    = "Gorilla_orth",
  Macaque    = "Macaque_orth",
  Marmoset   = "Marmoset_orth"
)

# 2. Create a TRUE/FALSE matrix of physical sequence presence
ortho_mat <- data.frame(peak_id = master_anno$peak_id)
rownames(ortho_mat) <- ortho_mat$peak_id

for (sp in names(ortho_cols)) {
  col_name <- ortho_cols[[sp]]
  ortho_mat[[sp]] <- as.logical(master_anno[[col_name]])
}

# 3. Define the Full Suite of Evolutionary Contrasts
ILS_contrasts <- list(
  # --- ILS / Topology Discordance (African Apes) ---
  "ILS_HumanGorilla_vs_Pan"         = list(pos = c("Human", "Gorilla"), neg = c("Chimpanzee", "Bonobo")),
  "ILS_HumanChimp_vs_GorillaBonobo" = list(pos = c("Human", "Chimpanzee"), neg = c("Gorilla", "Bonobo")),
  "ILS_HumanBonobo_vs_ChimpGorilla" = list(pos = c("Human", "Bonobo"), neg = c("Chimpanzee", "Gorilla")),
  
  # --- Single-Species Divergence ---
  "Div_Human_vs_Apes"        = list(pos = c("Human"), neg = c("Chimpanzee", "Bonobo", "Gorilla")),
  "Div_Chimp_vs_Apes"        = list(pos = c("Chimpanzee"), neg = c("Human", "Bonobo", "Gorilla")),
  "Div_Bonobo_vs_Apes"       = list(pos = c("Bonobo"), neg = c("Human", "Chimpanzee", "Gorilla")),
  "Div_Gorilla_vs_Apes"      = list(pos = c("Gorilla"), neg = c("Human", "Chimpanzee", "Bonobo")),
  "Div_Macaque_vs_GreatApes" = list(pos = c("Macaque"), neg = c("Human", "Chimpanzee", "Bonobo", "Gorilla")),
  
  # --- Deep Clade Nodes (Evolutionary Steps) ---
  "Node1_Human_vs_Pan"           = list(pos = c("Human"), neg = c("Chimpanzee", "Bonobo")),
  "Node2_AfricanApes_vs_Gorilla" = list(pos = c("Human", "Chimpanzee", "Bonobo"), neg = c("Gorilla")),
  "Node3_GreatApes_vs_Macaque"   = list(pos = c("Human", "Chimpanzee", "Bonobo", "Gorilla"), neg = c("Macaque")),
  "Node4_OldWorld_vs_Marmoset"   = list(pos = c("Human", "Chimpanzee", "Bonobo", "Gorilla", "Macaque"), neg = c("Marmoset")),
  
  # --- Pairwise Direct Comparisons ---
  "Pair_Human_vs_Gorilla"  = list(pos = c("Human"), neg = c("Gorilla")),
  "Pair_Human_vs_Chimp"    = list(pos = c("Human"), neg = c("Chimpanzee")),
  "Pair_Human_vs_Bonobo"   = list(pos = c("Human"), neg = c("Bonobo")),
  "Pair_Human_vs_Macaque"  = list(pos = c("Human"), neg = c("Macaque")),
  "Pair_Human_vs_Marmoset" = list(pos = c("Human"), neg = c("Marmoset"))
)

# 4. Dynamically generate the valid peak lists
node_valid_peaks <- list()
for (node in names(ILS_contrasts)) {
  req_species <- c(ILS_contrasts[[node]]$pos, ILS_contrasts[[node]]$neg)
  
  is_valid <- rowSums(ortho_mat[, req_species, drop = FALSE]) == length(req_species)
  is_valid[is.na(is_valid)] <- FALSE 
  
  node_valid_peaks[[node]] <- ortho_mat$peak_id[is_valid]
  
  message(sprintf("  [%s] %d peaks have orthology in required species: %s", 
                  node, length(node_valid_peaks[[node]]), paste(req_species, collapse=", ")))
}

message("✅ Dynamic Orthology indexing complete!")


=== Building Dynamic Orthology Index ===

  [ILS_HumanGorilla_vs_Pan] 955421 peaks have orthology in required species: Human, Gorilla, Chimpanzee, Bonobo

  [ILS_HumanChimp_vs_GorillaBonobo] 955421 peaks have orthology in required species: Human, Chimpanzee, Gorilla, Bonobo

  [ILS_HumanBonobo_vs_ChimpGorilla] 955421 peaks have orthology in required species: Human, Bonobo, Chimpanzee, Gorilla

  [Div_Human_vs_Apes] 955421 peaks have orthology in required species: Human, Chimpanzee, Bonobo, Gorilla

  [Div_Chimp_vs_Apes] 955421 peaks have orthology in required species: Chimpanzee, Human, Bonobo, Gorilla

  [Div_Bonobo_vs_Apes] 955421 peaks have orthology in required species: Bonobo, Human, Chimpanzee, Gorilla

  [Div_Gorilla_vs_Apes] 955421 peaks have orthology in required species: Gorilla, Human, Chimpanzee, Bonobo

  [Div_Macaque_vs_GreatApes] 904033 peaks have orthology in required species: Macaque, Human, Chimpanzee, Bonobo, Gorilla

  [Node1_Human_vs_Pan] 978300 peaks have ortholo

In [67]:
# =============================================================================
# Cell 16a: Prepare Unbiased Peaks for Evolutionary Testing
# =============================================================================
message("\n=== Generating Unbiased Peak Sets for Branch Testing ===")

evo_pb_counts <- list()

for (ct in unique(meta_shared$cell_type)) {
  ct_samples <- rownames(meta_shared)[meta_shared$cell_type == ct]
  ct_counts <- counts_shared[, ct_samples, drop = FALSE]
  
  # 👈 THE FIX: Light filter. Just require the peak to be active (>=10 counts) 
  # in at least 2 donors total. This perfectly preserves species-specific gains!
  keep_peaks <- rowSums(ct_counts >= 10) >= 2
  
  evo_pb_counts[[ct]] <- ct_counts[keep_peaks, , drop = FALSE]
  message(sprintf("  [%s] Kept %d unbiased peaks", ct, sum(keep_peaks)))
}

# Save this so we have it safely stored
saveRDS(evo_pb_counts, file.path(OUT_DIR, "pseudobulk", "evo_pb_counts.rds"))
message("✅ Unbiased evolutionary counts saved!")


=== Generating Unbiased Peak Sets for Branch Testing ===

  [Colonocytes] Kept 199492 unbiased peaks

  [Crypt.Fibroblasts.WNT2B.] Kept 152064 unbiased peaks

  [ECs] Kept 34636 unbiased peaks

  [Enterocytes] Kept 366077 unbiased peaks

  [Goblet.cells] Kept 371050 unbiased peaks

  [Macrophages] Kept 117679 unbiased peaks

  [Naive.B.cells] Kept 87699 unbiased peaks

  [Plasma.B.cells] Kept 129058 unbiased peaks

  [Specialized.Fibroblasts.RSPO3..only] Kept 65749 unbiased peaks

  [Specialized.Fibroblasts.SYNM.] Kept 297718 unbiased peaks

  [Stem.cells] Kept 155854 unbiased peaks

  [T.cells] Kept 220924 unbiased peaks

  [TA.cells] Kept 750138 unbiased peaks

✅ Unbiased evolutionary counts saved!



In [68]:
# =============================================================================
# Cell 22: Orthology-Aware ILS Testing (Standard & Ultra-Robust)
# =============================================================================
library(DESeq2)
library(matrixStats) # Required for the Ultra-Robust rowMins/rowMaxs

message("\n=== Running Orthology-Aware ILS & Pairwise Testing ===")

# Create separate output directories to keep things organized
std_dir <- file.path(OUT_DIR, "differential_results", "ILS_Standard")
ur_dir  <- file.path(OUT_DIR, "differential_results", "ILS_UltraRobust")
dir.create(std_dir, showWarnings = FALSE, recursive = TRUE)
dir.create(ur_dir, showWarnings = FALSE, recursive = TRUE)

# Contrast Helper Function
make_contrast_vector <- function(sp_pos, sp_neg, res_names) {
  vec <- rep(0, length(res_names))
  names(vec) <- res_names
  pos_names <- paste0("species", sp_pos)
  neg_names <- paste0("species", sp_neg)
  valid_pos <- intersect(pos_names, res_names)
  valid_neg <- intersect(neg_names, res_names)
  if (length(valid_pos) > 0) vec[valid_pos] <- 1 / length(valid_pos)
  if (length(valid_neg) > 0) vec[valid_neg] <- -1 / length(valid_neg)
  return(vec)
}

# Storage lists
ils_res_list <- list()
ur_peaks_list <- list() 

# Loop over Cell Types
for (ct in names(evo_pb_counts)) {
  message("\nProcessing cell type: ", ct)
  ils_res_list[[ct]] <- list()
  ur_peaks_list[[ct]] <- list()
  
  full_counts <- evo_pb_counts[[ct]]
  full_meta <- pb_meta[pb_meta$cell_type == ct, ]
  
  # Loop over every Contrast defined in Cell 21
  for (node in names(ILS_contrasts)) {
    req_species <- c(ILS_contrasts[[node]]$pos, ILS_contrasts[[node]]$neg)
    pos_sp <- ILS_contrasts[[node]]$pos
    neg_sp <- ILS_contrasts[[node]]$neg
    
    # Check if we have the species data in this specific cell type
    if (!any(pos_sp %in% full_meta$species) || !any(neg_sp %in% full_meta$species)) {
      message("  ⏭️ Skipping ", node, " (Missing species data)")
      next
    }
    
    # Subset to ONLY the peaks with physical sequence orthology
    valid_peaks_for_ct <- intersect(rownames(full_counts), node_valid_peaks[[node]])
    
    # Subset to ONLY the species involved in this specific comparison
    node_samples <- rownames(full_meta)[full_meta$species %in% req_species]
    
    counts <- full_counts[valid_peaks_for_ct, node_samples, drop = FALSE]
    meta <- full_meta[node_samples, ]
    meta$species <- factor(meta$species) 
    
    # Active peak filter: Must have signal (>=10) in at least 2 samples within this subset
    keep <- rowSums(counts >= 10) >= 2
    counts <- counts[keep, , drop = FALSE]
    
    # Ensure enough data remains to run a model
    if (nrow(counts) < 20 || length(unique(meta$species)) < 2) next
    
    # ---------------------------------------------------------
    # PART 1: Standard DESeq2 Testing
    # ---------------------------------------------------------
    dds <- DESeqDataSetFromMatrix(countData = counts, colData = meta, design = ~ 0 + species)
    dds <- estimateSizeFactors(dds, type = "poscounts")
    dds <- suppressMessages(DESeq(dds))
    
    res_names <- resultsNames(dds)
    contrast_vec <- make_contrast_vector(pos_sp, neg_sp, res_names)
    
    res <- results(dds, contrast = contrast_vec, alpha = 0.05)
    res_df <- as.data.frame(res)
    ils_res_list[[ct]][[node]] <- res_df
    
    # Identify Standard Gained Peaks
    std_up <- rownames(res_df)[!is.na(res_df$padj) & res_df$padj < 0.05 & res_df$log2FoldChange > 1]
    
    # ---------------------------------------------------------
    # PART 2: Ultra-Robust Perfect Separation Filter
    # ---------------------------------------------------------
    ur_up <- c() # Default to empty
    
    # Only run the robust filter if DESeq2 found anything AND we have enough donors
    pos_samples <- rownames(meta)[meta$species %in% pos_sp]
    neg_samples <- rownames(meta)[meta$species %in% neg_sp]
    
    if (length(std_up) > 0 && length(pos_samples) > 0 && length(neg_samples) > 0) {
      
      # Normalize counts to Log2CPM just for the significant peaks
      lib_sizes <- colSums(counts)
      cpm <- t(t(counts[std_up, , drop = FALSE]) / lib_sizes) * 1e6
      log_cpm <- log2(cpm + 1)
      
      # Split matrices
      pos_mat <- log_cpm[, pos_samples, drop = FALSE]
      neg_mat <- log_cpm[, neg_samples, drop = FALSE]
      
      # Apply the brutal filter: min(POS donors) > max(NEG donors)
      min_pos <- rowMins(as.matrix(pos_mat))
      max_neg <- rowMaxs(as.matrix(neg_mat))
      
      is_ur <- (min_pos > max_neg) & (min_pos > 1)
      is_ur[is.na(is_ur)] <- FALSE
      
      ur_up <- std_up[is_ur]
      ur_peaks_list[[ct]][[node]] <- ur_up
    }
    
    # ---------------------------------------------------------
    # PART 3: Logging and Saving
    # ---------------------------------------------------------
    message(sprintf("  ✅ %-32s: Standard = %-4d | Ultra-Robust = %d", 
                    node, length(std_up), length(ur_up)))
    
    # Save Standard Results
    if (length(std_up) > 0) {
      ct_std_dir <- file.path(std_dir, ct)
      dir.create(ct_std_dir, showWarnings = FALSE)
      write.csv(res_df[std_up, ], file.path(ct_std_dir, paste0(node, "_Standard_UP.csv")))
    }
    
    # Save Ultra-Robust Results (Includes the DESeq2 stats + Log2CPM values)
    if (length(ur_up) > 0) {
      ct_ur_dir <- file.path(ur_dir, ct)
      dir.create(ct_ur_dir, showWarnings = FALSE)
      
      # Combine stats with LogCPM matrix for easy reviewing
      ur_out <- cbind(res_df[ur_up, ], log_cpm[ur_up, , drop = FALSE])
      write.csv(ur_out, file.path(ct_ur_dir, paste0(node, "_UltraRobust_UP.csv")))
    }
  }
}

# Save the full lists to R objects
saveRDS(ils_res_list, file.path(std_dir, "ILS_Standard_DESeq2.rds"))
saveRDS(ur_peaks_list, file.path(ur_dir, "ILS_UltraRobust_Peaks.rds"))

message("\n🎉 Dual-Layer Differential Testing Complete!")


=== Running Orthology-Aware ILS & Pairwise Testing ===


Processing cell type: Colonocytes

  ✅ ILS_HumanGorilla_vs_Pan         : Standard = 654  | Ultra-Robust = 482

  ✅ ILS_HumanChimp_vs_GorillaBonobo : Standard = 597  | Ultra-Robust = 278

  ✅ ILS_HumanBonobo_vs_ChimpGorilla : Standard = 1293 | Ultra-Robust = 684

  ✅ Div_Human_vs_Apes               : Standard = 6443 | Ultra-Robust = 3652

  ✅ Div_Chimp_vs_Apes               : Standard = 62   | Ultra-Robust = 53

  ✅ Div_Bonobo_vs_Apes              : Standard = 639  | Ultra-Robust = 373

  ✅ Div_Gorilla_vs_Apes             : Standard = 563  | Ultra-Robust = 494

  ✅ Div_Macaque_vs_GreatApes        : Standard = 6659 | Ultra-Robust = 4653

  ✅ Node1_Human_vs_Pan              : Standard = 2899 | Ultra-Robust = 2317

  ✅ Node2_AfricanApes_vs_Gorilla    : Standard = 240  | Ultra-Robust = 240

  ✅ Node3_GreatApes_vs_Macaque      : Standard = 3251 | Ultra-Robust = 2881

  ✅ Node4_OldWorld_vs_Marmoset      : Standard = 5996 | Ultra-Robust

In [70]:
# =============================================================================
# Cell 23: The Perfect Evolutionary Waterfall (Strict Hierarchical Filtering)
# =============================================================================
suppressPackageStartupMessages({
  library(ComplexHeatmap)
  library(circlize)
  library(matrixStats)
})

message("\n=== Generating Strict Orthology-Aware Waterfall ===")

TARGET_CT <- "Enterocytes" # 👈 Change to your favorite cell type

# 1. Prepare raw counts and metadata
counts <- evo_pb_counts[[TARGET_CT]]
meta <- pb_meta[pb_meta$cell_type == TARGET_CT, ]
common_samples <- intersect(rownames(meta), colnames(counts))
counts <- counts[, common_samples, drop = FALSE]
meta <- meta[common_samples, ]

# 2. Normalize to Log2CPM
lib_sizes <- colSums(counts)
cpm <- t(t(counts) / lib_sizes) * 1e6
log_cpm <- log2(cpm + 1)

# 3. Calculate Average Expression per Species
species_order <- c("Human", "Chimpanzee", "Bonobo", "Gorilla", "Macaque", "Marmoset")
avg_exp <- matrix(NA, nrow = nrow(log_cpm), ncol = length(species_order))
rownames(avg_exp) <- rownames(log_cpm)
colnames(avg_exp) <- species_order

for (sp in species_order) {
  sp_samples <- rownames(meta)[meta$species == sp]
  if (length(sp_samples) > 1) {
    avg_exp[, sp] <- rowMeans(log_cpm[, sp_samples, drop = FALSE])
  } else if (length(sp_samples) == 1) {
    avg_exp[, sp] <- log_cpm[, sp_samples]
  }
}

# 4. Build the Orthology Mask
ortho_cols <- c(Human="Human_orth", Chimpanzee="Chimpanzee_orth", Bonobo="Bonobo_orth",
                Gorilla="Gorilla_orth", Macaque="Macaque_orth", Marmoset="Marmoset_orth")

ortho_mask <- matrix(TRUE, nrow = nrow(avg_exp), ncol = length(species_order))
rownames(ortho_mask) <- rownames(avg_exp)
colnames(ortho_mask) <- species_order
match_idx <- match(rownames(avg_exp), master_anno$peak_id)

for (sp in species_order) {
  col_name <- ortho_cols[[sp]]
  if (col_name %in% colnames(master_anno)) {
    ortho_mask[, sp] <- as.logical(master_anno[[col_name]][match_idx])
  }
}

# 5. THE MAGIC: Mathematical Hierarchical Filtering
# We create a math matrix where missing sequence is treated as -100 (infinitely silent)
math_exp <- avg_exp
math_exp[!ortho_mask] <- -100 

# Define clade minimums (lowest signal in the target group)
min_H     <- math_exp[, "Human"]
min_Afr   <- rowMins(math_exp[, c("Human", "Chimpanzee", "Bonobo")])
min_Great <- rowMins(math_exp[, c("Human", "Chimpanzee", "Bonobo", "Gorilla")])
min_OW    <- rowMins(math_exp[, c("Human", "Chimpanzee", "Bonobo", "Gorilla", "Macaque")])

# Define outgroup maximums (highest signal in ALL older species)
max_non_H     <- rowMaxs(math_exp[, c("Chimpanzee", "Bonobo", "Gorilla", "Macaque", "Marmoset")])
max_non_Afr   <- rowMaxs(math_exp[, c("Gorilla", "Macaque", "Marmoset")])
max_non_Great <- rowMaxs(math_exp[, c("Macaque", "Marmoset")])
max_Marm      <- math_exp[, "Marmoset"]

# CLASSIFY: The clade MUST be active (> 1), and strictly higher than the outgroup (+ 1 logCPM buffer)
is_human_novel_seq <- ortho_mask[, "Human"] & rowSums(ortho_mask) == 1

is_H_gain     <- (min_H > 1) & (min_H > max_non_H + 1) & !is_human_novel_seq
is_Afr_gain   <- (min_Afr > 1) & (min_Afr > max_non_Afr + 1) & !is_H_gain
is_Great_gain <- (min_Great > 1) & (min_Great > max_non_Great + 1) & !is_Afr_gain & !is_H_gain
is_OW_gain    <- (min_OW > 1) & (min_OW > max_Marm + 1) & !is_Great_gain & !is_Afr_gain & !is_H_gain
is_Novel      <- (min_H > 1) & is_human_novel_seq

# 6. Extract the Top 50 peaks per category (sorted by separation margin)
get_top_peaks <- function(logical_mask, margin_vector, n = 50) {
  peaks <- rownames(math_exp)[logical_mask]
  if (length(peaks) == 0) return(c())
  margins <- margin_vector[logical_mask]
  names(margins) <- peaks
  return(head(names(margins)[order(margins, decreasing = TRUE)], n))
}

plot_peaks <- c()
row_splits <- c()

cats <- list(
  "OldWorld_Gain"   = get_top_peaks(is_OW_gain, min_OW - max_Marm),
  "GreatApe_Gain"   = get_top_peaks(is_Great_gain, min_Great - max_non_Great),
  "AfricanApe_Gain" = get_top_peaks(is_Afr_gain, min_Afr - max_non_Afr),
  "Human_Gain"      = get_top_peaks(is_H_gain, min_H - max_non_H),
  "Human_Novel_Seq" = get_top_peaks(is_Novel, min_H)
)

for (grp in names(cats)) {
  if (length(cats[[grp]]) > 0) {
    plot_peaks <- c(plot_peaks, cats[[grp]])
    row_splits <- c(row_splits, rep(grp, length(cats[[grp]])))
  }
}

if(length(plot_peaks) == 0) stop("No peaks passed the strict hierarchical filter!")

# 7. Apply the Missing Sequence Mask to the actual data for plotting
avg_exp_masked <- avg_exp
avg_exp_masked[!ortho_mask] <- NA

# Safe Z-scoring (Handles NAs perfectly)
safe_scale <- function(x) {
  x_valid <- x[!is.na(x)]
  if (length(x_valid) > 1 && sd(x_valid) > 0) {
    x_scaled <- x
    x_scaled[!is.na(x)] <- (x_valid - mean(x_valid)) / sd(x_valid)
    return(x_scaled)
  } else if (length(x_valid) == 1) {
    x_scaled <- x
    x_scaled[!is.na(x)] <- 2 # Force novel sequences to red
    return(x_scaled)
  } else {
    return(rep(NA, length(x)))
  }
}

mat <- avg_exp_masked[plot_peaks, , drop = FALSE]
mat_scaled <- t(apply(mat, 1, safe_scale))
colnames(mat_scaled) <- colnames(mat)

# 8. Draw the Heatmap
split_factor <- factor(row_splits, levels = names(cats))
col_fun <- colorRamp2(c(-2, 0, 2), c("#377eb8", "white", "#e41a1c"))

ht <- Heatmap(mat_scaled, 
              name = "Z-score", 
              col = col_fun,
              na_col = "#d9d9d9", # 👈 THE MAGIC: Missing sequences become solid grey blocks!
              row_split = split_factor, 
              row_title_rot = 0, 
              row_title_gp = gpar(fontsize = 10, fontface = "bold"),
              row_gap = unit(3, "mm"), 
              cluster_rows = TRUE, 
              cluster_columns = FALSE, 
              show_row_names = FALSE, 
              column_names_rot = 45, 
              column_title = paste("Strict Evolutionary Waterfall -", TARGET_CT),
              column_title_gp = gpar(fontsize = 14, fontface = "bold"), 
              border = TRUE)

plot_dir <- file.path(OUT_DIR, "plots", "evolutionary_branches")
dir.create(plot_dir, showWarnings = FALSE, recursive = TRUE)
plot_file <- file.path(plot_dir, paste0(TARGET_CT, "_Strict_Waterfall.pdf"))

pdf(plot_file, width = 8, height = 12)
draw(ht, merge_legend = TRUE)
dev.off()

message("✅ Strict Waterfall Heatmap saved to: ", plot_file)


=== Generating Strict Orthology-Aware Waterfall ===



pdf 
  2

✅ Strict Waterfall Heatmap saved to: /cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/13_deseq2_R_pseudobulk/plots/evolutionary_branches/Enterocytes_Strict_Waterfall.pdf



In [74]:
# =============================================================================
# Fast Recovery: Rebuild Unbiased Matrix (C++ Matrix Mode)
# =============================================================================
library(dplyr)
library(arrow)

message("\n=== Rebuilding True Unbiased Matrix (Fast Mode) ===")

# 1. Load Parquet Files
SPECIES <- c("Human", "Chimpanzee", "Bonobo", "Gorilla", "Macaque", "Marmoset")
all_counts <- list()

for (sp in SPECIES) {
  sp_dir <- file.path(QUANT_DIR, sp)
  counts_file <- file.path(sp_dir, "pseudobulk_counts.parquet")
  meta_file <- file.path(sp_dir, "pseudobulk_groups.parquet") 
  
  if (!file.exists(counts_file)) next
  
  counts <- as.data.frame(read_parquet(counts_file))
  rownames(counts) <- counts$region_id
  counts$region_id <- NULL
  
  meta <- as.data.frame(read_parquet(meta_file))
  colnames(counts) <- paste0(sp, "_", meta$label)
  
  # 👈 THE TRICK: Convert to native matrix immediately
  all_counts[[sp]] <- as.matrix(counts)
  message("Loaded ", sp, ": ", nrow(counts), " peaks.")
}

union_peaks <- Reduce(union, lapply(all_counts, rownames))
message("\nTotal unique peaks across all primates: ", length(union_peaks))

# 2. 🌟 FAST MATRIX PRE-ALLOCATION 🌟
message("Merging matrices (this will now take seconds)...")

counts_union_list <- lapply(all_counts, function(mat) {
  # Instantly create a matrix of pure zeros
  new_mat <- matrix(0, nrow = length(union_peaks), ncol = ncol(mat), 
                    dimnames = list(union_peaks, colnames(mat)))
  
  # Slot the existing data right into the matching rows
  common <- intersect(rownames(mat), union_peaks)
  new_mat[common, ] <- mat[common, ]
  
  return(new_mat)
})

# Bind them instantly
counts_union <- do.call(cbind, counts_union_list)

# 3. Subset and Build evo_pb_counts
valid_samples <- rownames(pb_meta)
# Ensure we only subset columns that actually exist to prevent out-of-bounds
counts_union <- counts_union[, intersect(valid_samples, colnames(counts_union)), drop = FALSE]

evo_pb_counts <- list()

for (ct in unique(pb_meta$cell_type)) {
  ct_samples <- rownames(pb_meta)[pb_meta$cell_type == ct]
  ct_samples <- intersect(ct_samples, colnames(counts_union))
  
  ct_counts <- counts_union[, ct_samples, drop = FALSE]
  
  # Light filter: Require the peak to have >=10 counts in at least 2 donors total.
  keep_peaks <- rowSums(ct_counts >= 10) >= 2
  
  evo_pb_counts[[ct]] <- ct_counts[keep_peaks, , drop = FALSE]
  message(sprintf("  [%s] Kept %d true unbiased peaks", ct, sum(keep_peaks)))
}

saveRDS(evo_pb_counts, file.path(OUT_DIR, "pseudobulk", "evo_pb_counts.rds"))
message("✅ FAST Unbiased evolutionary counts successfully recovered and saved!")


=== Rebuilding True Unbiased Matrix (Fast Mode) ===

Loaded Human: 1039336 peaks.

Loaded Chimpanzee: 1029053 peaks.

Loaded Bonobo: 992889 peaks.

Loaded Gorilla: 1013198 peaks.

Loaded Macaque: 982905 peaks.

Loaded Marmoset: 971722 peaks.


Total unique peaks across all primates: 1142003

Merging matrices (this will now take seconds)...

  [Colonocytes] Kept 252192 true unbiased peaks

  [Crypt.Fibroblasts.WNT2B.] Kept 183305 true unbiased peaks

  [ECs] Kept 45580 true unbiased peaks

  [Enterocytes] Kept 430702 true unbiased peaks

  [Goblet.cells] Kept 452203 true unbiased peaks

  [Macrophages] Kept 152394 true unbiased peaks

  [Naive.B.cells] Kept 113565 true unbiased peaks

  [Plasma.B.cells] Kept 171490 true unbiased peaks

  [Specialized.Fibroblasts.RSPO3..only] Kept 81761 true unbiased peaks

  [Specialized.Fibroblasts.SYNM.] Kept 361336 true unbiased peaks

  [Stem.cells] Kept 188031 true unbiased peaks

  [T.cells] Kept 290064 true unbiased peaks

  [TA.cells] Kept 9611

In [75]:
# =============================================================================
# Cell 24: Innovation Heatmap (Shared Regulatory Gains vs Novel Sequence Insertions)
# =============================================================================
suppressPackageStartupMessages({
  library(ComplexHeatmap)
  library(circlize)
  library(matrixStats)
})

message("\n=== Generating Sequence & Regulatory Innovation Heatmap ===")

TARGET_CT <- "Enterocytes" # 👈 Change to your cell type

# 1. Prepare data
counts <- evo_pb_counts[[TARGET_CT]]
meta <- pb_meta[pb_meta$cell_type == TARGET_CT, ]
common_samples <- intersect(rownames(meta), colnames(counts))
counts <- counts[, common_samples, drop = FALSE]
meta <- meta[common_samples, ]

lib_sizes <- colSums(counts)
cpm <- t(t(counts) / lib_sizes) * 1e6
log_cpm <- log2(cpm + 1)

species_order <- c("Human", "Chimpanzee", "Bonobo", "Gorilla", "Macaque", "Marmoset")
avg_exp <- matrix(NA, nrow = nrow(log_cpm), ncol = length(species_order))
rownames(avg_exp) <- rownames(log_cpm)
colnames(avg_exp) <- species_order

for (sp in species_order) {
  sp_samples <- rownames(meta)[meta$species == sp]
  if (length(sp_samples) > 1) {
    avg_exp[, sp] <- rowMeans(log_cpm[, sp_samples, drop = FALSE])
  } else if (length(sp_samples) == 1) {
    avg_exp[, sp] <- log_cpm[, sp_samples]
  }
}

# 2. Build the Orthology Mask
ortho_cols <- c(Human="Human_orth", Chimpanzee="Chimpanzee_orth", Bonobo="Bonobo_orth",
                Gorilla="Gorilla_orth", Macaque="Macaque_orth", Marmoset="Marmoset_orth")

ortho_mask <- matrix(TRUE, nrow = nrow(avg_exp), ncol = length(species_order))
rownames(ortho_mask) <- rownames(avg_exp)
colnames(ortho_mask) <- species_order
match_idx <- match(rownames(avg_exp), master_anno$peak_id)

for (sp in species_order) {
  if (ortho_cols[[sp]] %in% colnames(master_anno)) {
    ortho_mask[, sp] <- as.logical(master_anno[[ortho_cols[[sp]]]][match_idx])
  }
}

# 3. DEFINE EXPLICIT ORTHOLOGY CLADES (This forces the Grey blocks!)
OWM_cols <- c("Human", "Chimpanzee", "Bonobo", "Gorilla", "Macaque")
GA_cols  <- c("Human", "Chimpanzee", "Bonobo", "Gorilla")
Afr_cols <- c("Human", "Chimpanzee", "Bonobo")
Pan_cols <- c("Chimpanzee", "Bonobo")

ortho_All      <- rowSums(ortho_mask) == 6
ortho_OWM_only <- rowSums(ortho_mask[, OWM_cols]) == 5 & !ortho_mask[, "Marmoset"]
ortho_GA_only  <- rowSums(ortho_mask[, GA_cols]) == 4  & !ortho_mask[, "Macaque"] & !ortho_mask[, "Marmoset"]
ortho_Afr_only <- rowSums(ortho_mask[, Afr_cols]) == 3 & !ortho_mask[, "Gorilla"] & !ortho_mask[, "Macaque"] & !ortho_mask[, "Marmoset"]
ortho_Hum_only <- ortho_mask[, "Human"] & rowSums(ortho_mask) == 1

# 4. DEFINE EXPRESSION RULES (Must be active > 1, and separated by > 0.5 logCPM)
# Group 1: Shared Sequences (Regulatory changes only)
is_Shared_OWM_Up  <- ortho_All & (rowMins(avg_exp[, OWM_cols, drop=FALSE]) > avg_exp[, "Marmoset"] + 0.5) & (rowMins(avg_exp[, OWM_cols, drop=FALSE]) > 1)
is_Shared_Marm_Up <- ortho_All & (avg_exp[, "Marmoset"] > rowMaxs(avg_exp[, OWM_cols, drop=FALSE]) + 0.5) & (avg_exp[, "Marmoset"] > 1)

# Group 2: Old World Novel Sequences (Grey in Marmoset)
is_SeqOWM_GA_Up  <- ortho_OWM_only & (rowMins(avg_exp[, GA_cols, drop=FALSE]) > avg_exp[, "Macaque"] + 0.5) & (rowMins(avg_exp[, GA_cols, drop=FALSE]) > 1)
is_SeqOWM_Mac_Up <- ortho_OWM_only & (avg_exp[, "Macaque"] > rowMaxs(avg_exp[, GA_cols, drop=FALSE]) + 0.5) & (avg_exp[, "Macaque"] > 1)

# Group 3: Great Ape Novel Sequences (Grey in Macaque & Marmoset)
is_SeqGA_Afr_Up <- ortho_GA_only & (rowMins(avg_exp[, Afr_cols, drop=FALSE]) > avg_exp[, "Gorilla"] + 0.5) & (rowMins(avg_exp[, Afr_cols, drop=FALSE]) > 1)
is_SeqGA_Gor_Up <- ortho_GA_only & (avg_exp[, "Gorilla"] > rowMaxs(avg_exp[, Afr_cols, drop=FALSE]) + 0.5) & (avg_exp[, "Gorilla"] > 1)

# Group 4: African Ape Novel Sequences (Grey in Gorilla, Macaque, Marmoset)
is_SeqAfr_Hum_Up <- ortho_Afr_only & (avg_exp[, "Human"] > rowMaxs(avg_exp[, Pan_cols, drop=FALSE]) + 0.5) & (avg_exp[, "Human"] > 1)
is_SeqAfr_Pan_Up <- ortho_Afr_only & (rowMins(avg_exp[, Pan_cols, drop=FALSE]) > avg_exp[, "Human"] + 0.5) & (rowMins(avg_exp[, Pan_cols, drop=FALSE]) > 1)

# Group 5: Human Novel Sequences (Grey in EVERYTHING else)
is_SeqHum_Only <- ortho_Hum_only & (avg_exp[, "Human"] > 1)

# 5. Extract Top Peaks for Plotting
get_top_peaks <- function(mask, n = 30) {
  peaks <- rownames(avg_exp)[mask]
  if(length(peaks) == 0) return(c())
  # Sort by Human signal to make the gradients look nice
  vals <- avg_exp[peaks, "Human"]
  names(vals) <- peaks
  return(head(names(vals)[order(vals, decreasing = TRUE)], n))
}

cats <- list(
  "Shared_OWM_Reg_Up"   = get_top_peaks(is_Shared_OWM_Up),
  "Shared_Marm_Reg_Up"  = get_top_peaks(is_Shared_Marm_Up),
  "Seq_OWM: GreatApe_Up"= get_top_peaks(is_SeqOWM_GA_Up),
  "Seq_OWM: Macaque_Up" = get_top_peaks(is_SeqOWM_Mac_Up),
  "Seq_GreatApe: Afr_Up"= get_top_peaks(is_SeqGA_Afr_Up),
  "Seq_GreatApe: Gor_Up"= get_top_peaks(is_SeqGA_Gor_Up),
  "Seq_AfrApe: Hum_Up"  = get_top_peaks(is_SeqAfr_Hum_Up),
  "Seq_AfrApe: Pan_Up"  = get_top_peaks(is_SeqAfr_Pan_Up),
  "Seq_Human_Specific"  = get_top_peaks(is_SeqHum_Only)
)

plot_peaks <- c()
row_splits <- c()
for (grp in names(cats)) {
  if (length(cats[[grp]]) > 0) {
    plot_peaks <- c(plot_peaks, cats[[grp]])
    row_splits <- c(row_splits, rep(grp, length(cats[[grp]])))
  }
}

if(length(plot_peaks) == 0) stop("No peaks passed the innovation filter!")

# 6. Apply Grey Mask and Z-score
avg_exp_masked <- avg_exp
avg_exp_masked[!ortho_mask] <- NA

safe_scale <- function(x) {
  x_val <- x[!is.na(x)]
  if (length(x_val) > 1 && sd(x_val) > 0) {
    x_sc <- x
    x_sc[!is.na(x)] <- (x_val - mean(x_val)) / sd(x_val)
    return(x_sc)
  } else if (length(x_val) == 1) {
    x_sc <- x
    x_sc[!is.na(x)] <- 2 # Force solitary values to Red
    return(x_sc)
  } else return(rep(NA, length(x)))
}

mat_scaled <- t(apply(avg_exp_masked[plot_peaks, , drop = FALSE], 1, safe_scale))
colnames(mat_scaled) <- colnames(avg_exp)

# 7. Draw the Heatmap
split_factor <- factor(row_splits, levels = names(cats))
col_fun <- colorRamp2(c(-2, 0, 2), c("#377eb8", "white", "#e41a1c"))

ht <- Heatmap(mat_scaled, 
              name = "Z-score", 
              col = col_fun,
              na_col = "#d9d9d9", # 👈 Grey blocks for missing sequences!
              row_split = split_factor, 
              row_title_rot = 0, 
              row_title_gp = gpar(fontsize = 9, fontface = "bold"),
              row_gap = unit(3, "mm"), 
              cluster_rows = TRUE, cluster_columns = FALSE, 
              show_row_names = FALSE, column_names_rot = 45, 
              column_title = paste("Sequence vs Regulatory Innovations -", TARGET_CT),
              column_title_gp = gpar(fontsize = 14, fontface = "bold"), border = TRUE)

plot_dir <- file.path(OUT_DIR, "plots", "evolutionary_branches")
dir.create(plot_dir, showWarnings = FALSE, recursive = TRUE)
plot_file <- file.path(plot_dir, paste0(TARGET_CT, "_Innovation_Heatmap.pdf"))

pdf(plot_file, width = 9, height = 13)
draw(ht, merge_legend = TRUE)
dev.off()

message("✅ Innovation Heatmap saved to: ", plot_file)


=== Generating Sequence & Regulatory Innovation Heatmap ===



pdf 
  2

✅ Innovation Heatmap saved to: /cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/13_deseq2_R_pseudobulk/plots/evolutionary_branches/Enterocytes_Innovation_Heatmap.pdf



In [ ]:
# =============================================================================
# Cell 24: Statistically-Powered Innovation Heatmap (DESeq2 + Ultra-Robust)
# =============================================================================
suppressPackageStartupMessages({
  library(ComplexHeatmap)
  library(circlize)
})

message("\n=== Generating Statistically-Powered Innovation Heatmap ===")

TARGET_CT <- "Enterocytes" # 👈 Change to your target cell type

if (!TARGET_CT %in% names(ur_peaks_list)) {
  stop("Target cell type not found in Ultra-Robust DESeq2 results. Did you run Cell 22?")
}

# 1. Prepare data
counts <- evo_pb_counts[[TARGET_CT]]
meta <- pb_meta[pb_meta$cell_type == TARGET_CT, ]
common_samples <- intersect(rownames(meta), colnames(counts))
counts <- counts[, common_samples, drop = FALSE]
meta <- meta[common_samples, ]

lib_sizes <- colSums(counts)
cpm <- t(t(counts) / lib_sizes) * 1e6
log_cpm <- log2(cpm + 1)

species_order <- c("Human", "Chimpanzee", "Bonobo", "Gorilla", "Macaque", "Marmoset")
avg_exp <- matrix(NA, nrow = nrow(log_cpm), ncol = length(species_order))
rownames(avg_exp) <- rownames(log_cpm)
colnames(avg_exp) <- species_order

for (sp in species_order) {
  sp_samples <- rownames(meta)[meta$species == sp]
  if (length(sp_samples) > 1) {
    avg_exp[, sp] <- rowMeans(log_cpm[, sp_samples, drop = FALSE])
  } else if (length(sp_samples) == 1) {
    avg_exp[, sp] <- log_cpm[, sp_samples]
  }
}

# 2. Build the Orthology Mask from master_anno
ortho_cols <- c(Human="Human_orth", Chimpanzee="Chimpanzee_orth", Bonobo="Bonobo_orth",
                Gorilla="Gorilla_orth", Macaque="Macaque_orth", Marmoset="Marmoset_orth")

ortho_mask <- matrix(TRUE, nrow = nrow(avg_exp), ncol = length(species_order))
rownames(ortho_mask) <- rownames(avg_exp)
colnames(ortho_mask) <- species_order
match_idx <- match(rownames(avg_exp), master_anno$peak_id)

for (sp in species_order) {
  if (ortho_cols[[sp]] %in% colnames(master_anno)) {
    ortho_mask[, sp] <- as.logical(master_anno[[ortho_cols[[sp]]]][match_idx])
  }
}

# 3. Categorize the STATISTICALLY SIGNIFICANT peaks from Cell 22
plot_peaks <- c()
row_splits <- c()

# Helper function to categorize a peak list based on physical DNA presence
categorize_ur_peaks <- function(ur_peaks, target_node_name) {
  if (length(ur_peaks) == 0) return(NULL)
  
  # Check sequence existence for these specific peaks
  peak_ortho <- ortho_mask[ur_peaks, , drop = FALSE]
  
  # Shared: DNA exists in Marmoset (so it was present before the Catarrhini split)
  is_shared <- peak_ortho[, "Marmoset"] == TRUE
  
  # Novel: DNA does NOT exist in the outgroups
  is_novel <- !is_shared
  
  # Top 50 highest logCPM in Human for clean plotting
  shared_peaks <- ur_peaks[is_shared]
  if (length(shared_peaks) > 50) {
      shared_peaks <- names(sort(avg_exp[shared_peaks, "Human"], decreasing = TRUE))[1:50]
  }
  
  novel_peaks <- ur_peaks[is_novel]
  if (length(novel_peaks) > 50) {
      novel_peaks <- names(sort(avg_exp[novel_peaks, "Human"], decreasing = TRUE))[1:50]
  }
  
  return(list(
    shared = shared_peaks,
    novel  = novel_peaks
  ))
}

# Iterate through your DESeq2 nodes
node_mapping <- list(
  "Node4_OldWorld_vs_Marmoset"   = "Old World Monkey",
  "Node3_GreatApes_vs_Macaque"   = "Great Ape",
  "Node2_AfricanApes_vs_Gorilla" = "African Ape",
  "Node1_Human_vs_Pan"           = "Human (Post-Pan)"
)

for (node_key in names(node_mapping)) {
  if (node_key %in% names(ur_peaks_list[[TARGET_CT]])) {
    cat_res <- categorize_ur_peaks(ur_peaks_list[[TARGET_CT]][[node_key]], node_mapping[[node_key]])
    
    if (length(cat_res$shared) > 0) {
      plot_peaks <- c(plot_peaks, cat_res$shared)
      row_splits <- c(row_splits, rep(paste("Shared Seq:", node_mapping[[node_key]], "Gain"), length(cat_res$shared)))
    }
    
    if (length(cat_res$novel) > 0) {
      plot_peaks <- c(plot_peaks, cat_res$novel)
      row_splits <- c(row_splits, rep(paste("Novel Seq:", node_mapping[[node_key]], "Insertion"), length(cat_res$novel)))
    }
  }
}

# Add Human-Only Unique Sequences (Which DESeq2 can't test because there's no comparison group)
is_human_only_seq <- ortho_mask[, "Human"] & rowSums(ortho_mask) == 1
human_only_peaks <- rownames(ortho_mask)[is_human_only_seq]
human_only_peaks <- human_only_peaks[human_only_peaks %in% rownames(avg_exp)]

if (length(human_only_peaks) > 0) {
  ho_exp <- avg_exp[human_only_peaks, "Human"]
  ho_top <- names(ho_exp)[order(ho_exp, decreasing = TRUE)]
  ho_top <- head(ho_top[ho_exp[ho_top] > 1], 50) # Must have logCPM > 1
  
  if (length(ho_top) > 0) {
    plot_peaks <- c(plot_peaks, ho_top)
    row_splits <- c(row_splits, rep("Novel Seq: Human Only Insertion", length(ho_top)))
  }
}

if(length(plot_peaks) == 0) stop("No peaks found to plot!")

# 4. Apply Grey Mask and Z-score
avg_exp_masked <- avg_exp
avg_exp_masked[!ortho_mask] <- NA

safe_scale <- function(x) {
  x_val <- x[!is.na(x)]
  if (length(x_val) > 1 && sd(x_val) > 0) {
    x_sc <- x
    x_sc[!is.na(x)] <- (x_val - mean(x_val)) / sd(x_val)
    return(x_sc)
  } else if (length(x_val) == 1) {
    x_sc <- x
    x_sc[!is.na(x)] <- 2
    return(x_sc)
  } else return(rep(NA, length(x)))
}

mat_scaled <- t(apply(avg_exp_masked[plot_peaks, , drop = FALSE], 1, safe_scale))
colnames(mat_scaled) <- colnames(avg_exp)

# 5. Draw the Heatmap
split_factor <- factor(row_splits, levels = unique(row_splits))
col_fun <- colorRamp2(c(-2, 0, 2), c("#377eb8", "white", "#e41a1c"))

ht <- Heatmap(mat_scaled, 
              name = "Z-score", 
              col = col_fun,
              na_col = "#d9d9d9", # Grey blocks for missing sequences!
              row_split = split_factor, 
              row_title_rot = 0, 
              row_title_gp = gpar(fontsize = 10, fontface = "bold"),
              row_gap = unit(3, "mm"), 
              cluster_rows = TRUE, cluster_columns = FALSE, 
              show_row_names = FALSE, column_names_rot = 45, 
              column_title = paste("Statistically-Powered Evolution -", TARGET_CT),
              column_title_gp = gpar(fontsize = 14, fontface = "bold"), border = TRUE)

plot_dir <- file.path(OUT_DIR, "plots", "evolutionary_branches")
dir.create(plot_dir, showWarnings = FALSE, recursive = TRUE)
plot_file <- file.path(plot_dir, paste0(TARGET_CT, "_Stat_Powered_Innovation.pdf"))

pdf(plot_file, width = 9, height = 13)
draw(ht, merge_legend = TRUE)
dev.off()

message("✅ Statistically-Powered Heatmap saved to: ", plot_file)

In [ ]:
# =============================================================================
# Cell 20: Parallel Motif Enrichment on Ultra-Robust Peaks
# =============================================================================
suppressPackageStartupMessages({
  library(monaLisa)
  library(JASPAR2022)
  library(TFBSTools)
  library(BSgenome.Hsapiens.UCSC.hg38)
  library(Biostrings)
  library(BiocParallel)
  library(ComplexHeatmap)
  library(circlize)
})

message("\n=== Running Parallel Motif Enrichment ===")

# Set up Parallel Backend (10 Cores)
mcparams <- BiocParallel::MulticoreParam(10L)
register(mcparams)
message("🚀 Parallel backend registered with 10 cores.")

motif_dir <- file.path(OUT_DIR, "differential_results", "motifs_robust")
dir.create(motif_dir, showWarnings = FALSE, recursive = TRUE)

pwms <- getMatrixSet(JASPAR2022, opts = list(tax_group = "vertebrates", matrixtype = "PWM"))

# 1. Parallel Enrichment Function
run_parallel_enrichment <- function(peak_list_named, pwms_obj, genome_obj, out_rds) {
  mapping_df <- data.frame(
    peak_id = unlist(peak_list_named, use.names = FALSE),
    group = rep(names(peak_list_named), lengths(peak_list_named))
  )
  
  coords <- get_peak_info(mapping_df$peak_id, "Human", "coordinates")
  mapping_df <- mapping_df[mapping_df$peak_id %in% coords$name, ]
  rownames(coords) <- coords$name
  coords <- coords[mapping_df$peak_id, ] 
  
  gr <- makeGRangesFromDataFrame(coords)
  seqs <- getSeq(genome_obj, gr)
  group_factor <- factor(mapping_df$group, levels = names(peak_list_named))
  
  message("Running parallel binned enrichment for ", length(levels(group_factor)), " groups...")
  
  se_enrich <- calcBinnedMotifEnrR(seqs = seqs, 
                                   bins = group_factor,
                                   pwmL = pwms_obj,       
                                   min.score = 10,
                                   BPPARAM = mcparams,   # 👈 Parallel execution!
                                   verbose = TRUE)
  
  saveRDS(se_enrich, out_rds)
  message("💾 Saved object to: ", basename(out_rds))
  return(se_enrich)
}

# -----------------------------------------------------------------------------
# Part A: Ultra-Robust Cell Types (Human vs All)
# -----------------------------------------------------------------------------
# Re-gather the robust cell type peaks we generated in Cell 18
message("\n--- Motif Enrichment: Ultra-Robust Cell Types ---")
ur_ct_peaks <- list()
for (ct in names(robust_summary)) {
  # Load the ultra-robust CSV we saved in Cell 18
  csv_file <- file.path(OUT_DIR, "differential_results", "ultra_robust_peaks", paste0(ct, "_Ultra_Robust_Human_UP.csv"))
  if (file.exists(csv_file)) {
    ur_df <- read.csv(csv_file)
    if (nrow(ur_df) >= 15) { # Only run motifs if we have enough peaks
      ur_ct_peaks[[ct]] <- ur_df$peak_id
    }
  }
}

if (length(ur_ct_peaks) > 0) {
  se_ur_ct <- run_parallel_enrichment(ur_ct_peaks, pwms, BSgenome.Hsapiens.UCSC.hg38, 
                                      file.path(motif_dir, "UltraRobust_CellType_SE.rds"))
}

# -----------------------------------------------------------------------------
# Part B: Ultra-Robust Branches (Focusing on Enterocytes)
# -----------------------------------------------------------------------------
# We'll plot the branch evolution specifically for Enterocytes as an example
message("\n--- Motif Enrichment: Ultra-Robust Evolutionary Branches ---")
TARGET_CT <- "Enterocytes"
ur_branch_peaks <- robust_branch_peaks[[TARGET_CT]]

# Filter out branches with too few peaks for reliable motif enrichment
ur_branch_peaks <- ur_branch_peaks[sapply(ur_branch_peaks, length) >= 15]

if (length(ur_branch_peaks) > 0) {
  se_ur_branch <- run_parallel_enrichment(ur_branch_peaks, pwms, BSgenome.Hsapiens.UCSC.hg38, 
                                          file.path(motif_dir, paste0(TARGET_CT, "_UltraRobust_Branches_SE.rds")))
}

# -----------------------------------------------------------------------------
# 3. Quick Plotting Function for the Results
# -----------------------------------------------------------------------------
plot_motif_heatmap <- function(se_obj, title, filename) {
  if (is.null(se_obj)) return()
  
  mat <- assay(se_obj, "negLog10Padj")
  rownames(mat) <- gsub(".*_", "", rownames(mat))
  
  top_indices <- unique(as.vector(apply(mat, 2, function(x) order(x, decreasing = TRUE)[1:3])))
  plot_sub <- mat[top_indices, , drop=FALSE]
  plot_sub[plot_sub > 20] <- 20 
  
  col_fun <- colorRamp2(c(0, 2, 20), c("white", "orange", "red"))
  
  ht <- Heatmap(plot_sub, name = "Enrichment", col = col_fun,
                column_title = title, cluster_rows = TRUE, cluster_columns = TRUE,
                show_row_names = TRUE, border = TRUE, column_names_rot = 45,
                heatmap_legend_param = list(title = "-log10(p-adj)"))
  
  pdf(filename, width = 10, height = 12)
  draw(ht, merge_legend = TRUE)
  dev.off()
}

if (exists("se_ur_ct")) plot_motif_heatmap(se_ur_ct, "Motifs in Ultra-Robust Human Peaks", file.path(OUT_DIR, "plots", "Motif_UltraRobust_CellTypes.pdf"))
if (exists("se_ur_branch")) plot_motif_heatmap(se_ur_branch, paste("Motifs in Evolutionary Branches -", TARGET_CT), file.path(OUT_DIR, "plots", paste0("Motif_UltraRobust_Branches_", TARGET_CT, ".pdf")))

message("\n🎉 Ultra-Robust Motif Enrichment finished rapidly thanks to parallelization!")

In [22]:
# =============================================================================
# Cell 13: Minimal Motif Enrichment (with Matrix Saving)
# =============================================================================
library(monaLisa)
library(JASPAR2022)
library(TFBSTools)
library(BSgenome.Hsapiens.UCSC.hg38)
library(ComplexHeatmap)
library(Biostrings)
library(circlize)

message("\n--- Loading Motif Database ---")
pwms <- getMatrixSet(JASPAR2022, opts = list(tax_group = "vertebrates", matrixtype = "PWM"))

# Ensure output directories exist
motif_dir <- file.path(OUT_DIR, "differential_results", "motifs")
dir.create(motif_dir, showWarnings = FALSE, recursive = TRUE)

# 1. The Core Enrichment Function
run_batch_motif <- function(peak_list_named, pwms_obj, genome_obj) {
  mapping_df <- data.frame(
    peak_id = unlist(peak_list_named, use.names = FALSE),
    group = rep(names(peak_list_named), lengths(peak_list_named))
  )
  
  # Fetch coordinates using your existing get_peak_info function
  coords <- get_peak_info(mapping_df$peak_id, "Human", "coordinates")
  mapping_df <- mapping_df[mapping_df$peak_id %in% coords$name, ]
  rownames(coords) <- coords$name
  coords <- coords[mapping_df$peak_id, ] 
  
  gr <- makeGRangesFromDataFrame(coords)
  seqs <- getSeq(genome_obj, gr)
  group_factor <- factor(mapping_df$group, levels = names(peak_list_named))
  
  message("Running binned enrichment for ", length(levels(group_factor)), " groups. This may take a moment...")
  
  se_enrich <- calcBinnedMotifEnrR(seqs = seqs, 
                                   bins = group_factor,
                                   pwmL = pwms_obj,       
                                   min.score = 10,        
                                   verbose = TRUE)
  
  enr_mat <- assay(se_enrich, "negLog10Padj")
  rownames(enr_mat) <- gsub(".*_", "", rownames(enr_mat)) # Clean JASPAR names
  return(enr_mat)
}

# -----------------------------------------------------------------------------
# Part A: Cell Type Motifs
# -----------------------------------------------------------------------------
message("\n--- Processing Cell Types ---")
ct_peaks <- list()
for (ct in names(res_list)) {
  if ("Human" %in% names(res_list[[ct]])) {
    res_df <- as.data.frame(res_list[[ct]][["Human"]])
    up_peaks <- rownames(res_df)[!is.na(res_df$padj) & res_df$padj < 0.05 & res_df$log2FoldChange > 1]
    if (length(up_peaks) >= 20) ct_peaks[[ct]] <- up_peaks
  }
}

ct_enr_mat <- run_batch_motif(ct_peaks, pwms, BSgenome.Hsapiens.UCSC.hg38)

# 👈 SAVE MATRICES IMMEDIATELY
write.csv(ct_enr_mat, file.path(motif_dir, "CellType_Motif_Enrichment_Matrix.csv"))
saveRDS(ct_enr_mat, file.path(motif_dir, "CellType_Motif_Enrichment_Matrix.rds"))
message("✅ Cell Type Motif Matrix saved!")

# -----------------------------------------------------------------------------
# Part B: Module Motifs
# -----------------------------------------------------------------------------
message("\n--- Processing Modules ---")
mod_peaks_list <- split(master_df$region_id, master_df$module)
mod_peaks_list <- mod_peaks_list[sapply(mod_peaks_list, length) >= 20]

mod_enr_mat <- run_batch_motif(mod_peaks_list, pwms, BSgenome.Hsapiens.UCSC.hg38)

# 👈 SAVE MATRICES IMMEDIATELY
write.csv(mod_enr_mat, file.path(motif_dir, "Module_Motif_Enrichment_Matrix.csv"))
saveRDS(mod_enr_mat, file.path(motif_dir, "Module_Motif_Enrichment_Matrix.rds"))
message("✅ Module Motif Matrix saved!")

Loading required package: BiocFileCache

Loading required package: dbplyr


Attaching package: ‘dbplyr’


The following objects are masked from ‘package:dplyr’:

    ident, sql


Loading required package: BSgenome

Loading required package: Biostrings

Loading required package: XVector


Attaching package: ‘Biostrings’


The following object is masked from ‘package:grid’:

    pattern


The following object is masked from ‘package:base’:

    strsplit


Loading required package: BiocIO

Loading required package: rtracklayer


Attaching package: ‘rtracklayer’


The following object is masked from ‘package:BiocIO’:

    FileForFormat



--- Loading Motif Database ---


--- Processing Cell Types ---

Running binned enrichment for 13 groups. This may take a moment...

Filtering sequences ...

  in total filtering out 0 of 38441 sequences (0%)

Scanning sequences for motif hits...

Create motif hit matrix...

starting analysis of bin Colonocytes

Defining background sequence set (otherBins)

ERROR: Error in grid.Call.graphics(C_downvppath, name$path, name$name, strict): Viewport '-log10(p-adj)_heatmap_body_1_1' was not found


In [23]:
# =============================================================================
# Cell 13.5: Draw Motif Heatmaps (from Saved Matrices)
# =============================================================================
library(ComplexHeatmap)
library(circlize)

message("\n--- Loading Saved Motif Matrices ---")
motif_dir <- file.path(OUT_DIR, "differential_results", "motifs")
ct_enr_mat <- readRDS(file.path(motif_dir, "CellType_Motif_Enrichment_Matrix.rds"))
mod_enr_mat <- readRDS(file.path(motif_dir, "Module_Motif_Enrichment_Matrix.rds"))

# Updated Plotting Function with the Viewport Fix
plot_motif_heatmap <- function(mat, title, filename) {
  # Get top 3 motifs per column
  top_indices <- unique(as.vector(apply(mat, 2, function(x) order(x, decreasing = TRUE)[1:3])))
  plot_sub <- mat[top_indices, ]
  plot_sub[plot_sub > 20] <- 20 # Cap significance
  
  col_fun <- colorRamp2(c(0, 2, 20), c("white", "orange", "red"))
  
  # Build the heatmap object first
  ht <- Heatmap(plot_sub, 
                name = "Enrichment",           # 👈 THE FIX: Safe internal name (no hyphens/parentheses)
                col = col_fun,
                column_title = title,
                cluster_rows = TRUE, 
                cluster_columns = TRUE,
                show_row_names = TRUE,
                border = TRUE,
                column_names_rot = 45,
                heatmap_legend_param = list(title = "-log10(p-adj)") # 👈 Move the special text here!
  )
  
  # Explicitly draw it into the PDF
  pdf(filename, width = 12, height = 14)
  draw(ht, merge_legend = TRUE)
  dev.off()
}

message("--- Plotting Heatmaps ---")
plot_motif_heatmap(ct_enr_mat, "Top Motifs per Human-UP Cell Type", 
                   file.path(OUT_DIR, "plots", "Motif_vs_CellType_Heatmap.pdf"))

plot_motif_heatmap(mod_enr_mat, "Top Motifs per K-Means Module", 
                   file.path(OUT_DIR, "plots", "Motif_vs_Module_Heatmap.pdf"))

message("✅ Motif heatmaps successfully drawn and saved!")


--- Loading Saved Motif Matrices ---

--- Plotting Heatmaps ---



pdf 
  2

pdf 
  2

✅ Motif heatmaps successfully drawn and saved!



In [24]:
# =============================================================================
# Cell 15: Native monaLisa Heatmaps with Sequence Logos
# =============================================================================
library(monaLisa)
library(JASPAR2022)
library(TFBSTools)
library(BSgenome.Hsapiens.UCSC.hg38)

message("\n=== Generating monaLisa Native Plots with Logos ===")

# Ensure motif database is loaded
pwms <- getMatrixSet(JASPAR2022, opts = list(tax_group = "vertebrates", matrixtype = "PWM"))

# Custom wrapper to run enrichment, filter top hits, and plot with logos
plot_monalisa_native <- function(peak_list_named, pwms_obj, genome_obj, filename, top_n = 30) {
  
  # 1. Fetch sequences
  mapping_df <- data.frame(
    peak_id = unlist(peak_list_named, use.names = FALSE),
    group = rep(names(peak_list_named), lengths(peak_list_named))
  )
  
  coords <- get_peak_info(mapping_df$peak_id, "Human", "coordinates")
  mapping_df <- mapping_df[mapping_df$peak_id %in% coords$name, ]
  rownames(coords) <- coords$name
  coords <- coords[mapping_df$peak_id, ] 
  
  gr <- makeGRangesFromDataFrame(coords)
  seqs <- getSeq(genome_obj, gr)
  group_factor <- factor(mapping_df$group, levels = names(peak_list_named))
  
  # 2. Run Enrichment (Generates the SummarizedExperiment object)
  message("Running enrichment for ", filename, "...")
  se <- calcBinnedMotifEnrR(seqs = seqs, 
                            bins = group_factor,
                            pwmL = pwms_obj,       
                            min.score = 10,        
                            verbose = FALSE)
  
  # 3. FILTER FOR TOP MOTIFS
  # Find the maximum enrichment score for each motif across all groups
  enr_mat <- assay(se, "negLog10Padj")
  max_enr <- apply(enr_mat, 1, max, na.rm = TRUE)
  
  # Keep only the top 'N' motifs to prevent plotting crashes
  top_motif_idx <- order(max_enr, decreasing = TRUE)[1:top_n]
  se_sub <- se[top_motif_idx, ]
  
  # 4. Draw the Native monaLisa Plot
  message("Drawing Heatmap with Logos to PDF...")
  
  # Width needs to be large to fit the Logos, GC content, Enrichment, and FDR panels!
  pdf(filename, width = 16, height = 12) 
  plotMotifHeatmaps(x = se_sub, 
                    which.plots = c("enr", "FDR"), 
                    cluster = TRUE,
                    show_motif_GC = TRUE,
                    show_seqlogo = TRUE)
  dev.off()
  
  message("✅ Saved to: ", filename)
  return(se) # Return the object just in case you want to explore it later
}

# -----------------------------------------------------------------------------
# Generate the Plots
# -----------------------------------------------------------------------------
# Assuming ct_peaks and mod_peaks_list are still in your environment from Cell 13
# If not, you can re-build them using the loops from the previous code block

if (exists("ct_peaks")) {
  se_ct <- plot_monalisa_native(ct_peaks, pwms, BSgenome.Hsapiens.UCSC.hg38, 
                                file.path(OUT_DIR, "plots", "Native_monaLisa_CellTypes.pdf"), 
                                top_n = 35)
}

if (exists("mod_peaks_list")) {
  se_mod <- plot_monalisa_native(mod_peaks_list, pwms, BSgenome.Hsapiens.UCSC.hg38, 
                                 file.path(OUT_DIR, "plots", "Native_monaLisa_Modules.pdf"), 
                                 top_n = 35)
}

message("🎉 All sequence logo heatmaps completed!")


=== Generating monaLisa Native Plots with Logos ===

Running enrichment for /cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/13_deseq2_R_pseudobulk/plots/Native_monaLisa_CellTypes.pdf...



In [ ]:
# =============================================================================
# Cell 13: Systematic Motif Enrichment (Aligned to monaLisa Tutorial)
# =============================================================================
library(monaLisa)
library(JASPAR2022)
library(TFBSTools)
library(BSgenome.Hsapiens.UCSC.hg38)
library(ComplexHeatmap)
library(Biostrings)

message("\n--- Loading Motif Database ---")
# Get Vertebrate PWMs just like the tutorial
pwms <- getMatrixSet(JASPAR2022, opts = list(tax_group = "vertebrates", matrixtype = "PWM"))

# 1. Helper Function
run_batch_motif <- function(peak_list_named, pwms_obj, genome_obj) {
  
  # A. Create mapping dataframe of Peak -> Group
  mapping_df <- data.frame(
    peak_id = unlist(peak_list_named, use.names = FALSE),
    group = rep(names(peak_list_named), lengths(peak_list_named))
  )
  
  # B. Get coordinates for ALL peaks
  coords <- get_peak_info(mapping_df$peak_id, "Human", "coordinates")
  mapping_df <- mapping_df[mapping_df$peak_id %in% coords$name, ]
  rownames(coords) <- coords$name
  coords <- coords[mapping_df$peak_id, ] 
  
  # C. Master GRanges and DNAStringSet
  gr <- makeGRangesFromDataFrame(coords)
  seqs <- getSeq(genome_obj, gr)
  
  # D. Factor for the bins
  group_factor <- factor(mapping_df$group, levels = names(peak_list_named))
  
  message("Running binned enrichment statistics for ", length(levels(group_factor)), " groups...")
  
  # E. Run Enrichment (Exact tutorial syntax!)
  se_enrich <- calcBinnedMotifEnrR(seqs = seqs, 
                                   bins = group_factor,
                                   pwmL = pwms_obj,       # 👈 Fixed: pwmL, not pwms
                                   min.score = 10,        # 👈 Added to speed up and reduce noise
                                   verbose = TRUE)
  
  # F. Extract -log10(p-adj)
  enr_mat <- assay(se_enrich, "negLog10Padj")
  
  # Clean up JASPAR names for the plot (e.g., "MA0114.1_HNF4A" -> "HNF4A")
  rownames(enr_mat) <- gsub(".*_", "", rownames(enr_mat))
  
  return(enr_mat)
}

# -----------------------------------------------------------------------------
# Part A: Motifs vs. Cell Types
# -----------------------------------------------------------------------------
message("\nProcessing Cell Type Motifs...")
ct_peaks <- list()
for (ct in names(res_list)) {
  if ("Human" %in% names(res_list[[ct]])) {
    res_df <- as.data.frame(res_list[[ct]][["Human"]])
    up_peaks <- rownames(res_df)[!is.na(res_df$padj) & res_df$padj < 0.05 & res_df$log2FoldChange > 1]
    if (length(up_peaks) >= 20) ct_peaks[[ct]] <- up_peaks
  }
}

ct_enr_mat <- run_batch_motif(ct_peaks, pwms, BSgenome.Hsapiens.UCSC.hg38)

# -----------------------------------------------------------------------------
# Part B: Motifs vs. Modules
# -----------------------------------------------------------------------------
message("\nProcessing Module Motifs...")
mod_peaks_list <- split(master_df$region_id, master_df$module)
mod_peaks_list <- mod_peaks_list[sapply(mod_peaks_list, length) >= 20]

mod_enr_mat <- run_batch_motif(mod_peaks_list, pwms, BSgenome.Hsapiens.UCSC.hg38)

# -----------------------------------------------------------------------------
# 3. Create Heatmaps
# -----------------------------------------------------------------------------
plot_motif_heatmap <- function(mat, title, filename) {
  # Keep only the top 3 driving motifs per column to prevent a massive unreadable plot
  top_indices <- unique(as.vector(apply(mat, 2, function(x) order(x, decreasing = TRUE)[1:3])))
  plot_sub <- mat[top_indices, ]
  
  # Cap significance at 20 so mega-hits don't wash out the subtle ones
  plot_sub[plot_sub > 20] <- 20
  
  col_fun <- colorRamp2(c(0, 2, 20), c("white", "orange", "red"))
  
  pdf(filename, width = 12, height = 14)
  print(Heatmap(plot_sub, 
                name = "-log10(p-adj)", 
                col = col_fun,
                column_title = title,
                cluster_rows = TRUE, 
                cluster_columns = TRUE,
                show_row_names = TRUE,
                border = TRUE,
                column_names_rot = 45))
  dev.off()
}

plot_motif_heatmap(ct_enr_mat, "Top Motifs per Human-UP Cell Type", 
                   file.path(OUT_DIR, "plots", "Motif_vs_CellType_Heatmap.pdf"))

plot_motif_heatmap(mod_enr_mat, "Top Motifs per K-Means Module", 
                   file.path(OUT_DIR, "plots", "Motif_vs_Module_Heatmap.pdf"))

message("\n✅ Motif heatmaps saved to: ", file.path(OUT_DIR, "plots"))


--- Loading Motif Database ---


Processing Cell Type Motifs...

Running binned enrichment statistics for 13 groups...

Filtering sequences ...

  in total filtering out 0 of 38441 sequences (0%)

Scanning sequences for motif hits...

